
# 2. DATASET CLEANING AND CONSTRUCTION

## 2.1 Introduction

This notebook focuses on cleaning, standardizing, selecting
and preparing agricultural question-and-answer records
for the Edo Language Agricultural Advisory Chatbot.

The datasets acquired and inspected in Notebook 01 will
serve as the initial sources of agricultural information.

Since the acquired datasets have different structures,
languages and levels of contextual information, they
must be processed into a consistent format.

The original datasets will remain unchanged. All cleaning,
transformation and selection operations will be performed
on working copies.

## 2.2 Objectives

The objectives of this stage are to:

1. Load the acquired agricultural datasets.
2. Standardize their structures and field names.
3. Identify incomplete, duplicated and irrelevant records.
4. Establish a consistent agricultural intent taxonomy.
5. Identify agricultural entities, including crop types
   and farming problems.
6. Assess the relevance of records to Nigerian agriculture.
7. Select at least 350 unique agricultural Q&A examples.
8. Prepare linked English and Edo (Bini) records.
9. Establish translation and annotation review metadata.
10. Prepare a separate collection of approved agricultural
    knowledge passages for retrieval.

## 2.3 Dataset Construction Principles

Each underlying agricultural question-and-answer example
will receive a unique identifier.

English and Edo versions of the same example will share
a translation group identifier and will not be counted
as separate underlying examples.

Records will retain their original source information
to support traceability and appropriate attribution.

Unverified translations, classifications and agricultural
recommendations will be explicitly marked as pending review.

The initial dataset target is at least 350 unique
agricultural question-and-answer examples.



## 2.4 Importing Required Libraries

This section imports the Python libraries required
for dataset preparation.

The following libraries will be used:

- Pandas: Data manipulation and dataset processing.
- NumPy: Numerical operations and missing-value handling.
- Pathlib: Project directory and file management.
- Regular Expressions: Text cleaning and pattern matching.
- JSON: Reading and writing structured dataset records.
- Hugging Face Datasets: Loading datasets previously
  saved to local storage.

These libraries will support the cleaning and
standardization operations performed in this notebook.


In [1]:

# Import libraries required for dataset preparation.

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

from datasets import load_from_disk

print("All required libraries imported successfully.")


c:\Users\XXMUGENX\3D Objects\edofarm-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required libraries imported successfully.



## 2.5 Project Directory Configuration

This section establishes the directory paths used
throughout the dataset construction process.

The project uses separate directories for:

- Raw data: Original downloaded datasets.
- Processed data: Cleaned and standardized records.
- Translations: English and Edo translation records.
- Knowledge base: Approved agricultural information
  intended for retrieval.
- Metadata: Dataset documentation and inspection reports.

Separating these directories helps preserve the original
data and maintain an organized workflow.


In [2]:

# Identify the project root directory.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

# Configure project directories.

DATA_DIR = PROJECT_DIR / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
TRANSLATIONS_DIR = DATA_DIR / "translations"
KNOWLEDGE_DIR = DATA_DIR / "knowledge_base"
METADATA_DIR = DATA_DIR / "metadata"

# Create output directories if they do not exist.

for directory in [
    PROCESSED_DIR,
    TRANSLATIONS_DIR,
    KNOWLEDGE_DIR,
    METADATA_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories configured successfully.")

print("\nProject root:")
print(PROJECT_DIR)

print("\nRaw data directory:")
print(RAW_DIR)

print("\nOutput directories:")
for directory in [
    PROCESSED_DIR,
    TRANSLATIONS_DIR,
    KNOWLEDGE_DIR,
    METADATA_DIR
]:
    print(directory)


Project directories configured successfully.

Project root:
c:\Users\XXMUGENX\3D Objects\edofarm-ai

Raw data directory:
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\raw

Output directories:
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\translations
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\knowledge_base
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata



## 2.6 Verification of Locally Stored Datasets

The original datasets were downloaded and saved locally
during the dataset inspection stage.

Before loading them, their directory locations will
be examined to confirm that the expected files exist.

This verification prevents unnecessary downloads and
ensures that subsequent processing uses the same
original datasets examined in 01_dataset_acquisition.ipynb.


In [3]:

# Display the contents of the raw data directory.

print("Raw data directory exists:", RAW_DIR.exists())

if RAW_DIR.exists():

    print("\nContents of data/raw:\n")

    for item in RAW_DIR.iterdir():

        item_type = "Directory" if item.is_dir() else "File"

        print(f"{item.name} - {item_type}")

else:

    print("Raw data directory was not found.")


Raw data directory exists: True

Contents of data/raw:

agronomy_qa - Directory
farmerchat_nigeria - Directory


## 3. Inspecting and Loading the Saved Datasets

The two datasets acquired during Notebook 01 are stored locally in `data/raw`:

- `farmerchat_nigeria`
- `agronomy_qa`

Before loading them, we will inspect their directory contents. This helps us determine whether they were saved as Hugging Face datasets, CSV files, Parquet files, or another format.

The original datasets will remain unchanged. All cleaning and transformation will be performed on separate working copies.

In [4]:
def inspect_directory(directory):
    print(f"\nContents of: {directory}")

    if not directory.exists():
        print("Directory not found.")
        return

    for item in sorted(directory.rglob("*")):
        if item.is_file():
            print(f"FILE: {item.relative_to(directory)}")


FARMERCHAT_DIR = RAW_DIR / "farmerchat_nigeria"
AGRONOMY_DIR = RAW_DIR / "agronomy_qa"

inspect_directory(FARMERCHAT_DIR)
inspect_directory(AGRONOMY_DIR)


Contents of: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\raw\farmerchat_nigeria
FILE: data-00000-of-00001.arrow
FILE: dataset_info.json
FILE: state.json

Contents of: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\raw\agronomy_qa
FILE: data-00000-of-00001.arrow
FILE: dataset_info.json
FILE: state.json


## 3. Loading the Original Datasets

The two datasets inspected in Notebook 01 are stored locally in
Hugging Face's Arrow format.

We will load them directly from the raw data directory.

The original datasets will remain unchanged. All cleaning,
standardisation, filtering, and transformation will be performed
on separate working copies.

Datasets:
- FarmerChat Nigeria
- Agronomy QA Pairs

In [5]:
# Load both datasets from their local directories

FARMERCHAT_PATH = RAW_DIR / "farmerchat_nigeria"
AGRONOMY_PATH = RAW_DIR / "agronomy_qa"

farmerchat_raw = load_from_disk(str(FARMERCHAT_PATH))
agronomy_raw = load_from_disk(str(AGRONOMY_PATH))

print("Datasets loaded successfully.")
print("\nFarmerChat Nigeria:")
print(farmerchat_raw)

print("\nAgronomy QA Pairs:")
print(agronomy_raw)

Datasets loaded successfully.

FarmerChat Nigeria:
Dataset({
    features: ['crop', 'user_country', 'user_geo_level2', 'user_geo_level3', 'query_year_month', 'query', 'response'],
    num_rows: 11541
})

Agronomy QA Pairs:
Dataset({
    features: ['question', 'answer'],
    num_rows: 1914
})


## 4. Inspecting Dataset Structures

The datasets may use different field names and structures.

We will inspect:
- Number of records
- Available columns
- Data types
- A sample record from each dataset

This information will guide the standardisation process.

In [6]:
def inspect_dataset(dataset, dataset_name):
    print("=" * 60)
    print(dataset_name)
    print("=" * 60)

    print("\nNumber of records:", len(dataset))
    print("\nAvailable columns:")

    for column in dataset.column_names:
        print(f"- {column}")

    print("\nDataset features:")
    print(dataset.features)

    print("\nSample record:")
    sample = dataset[0]

    for key, value in sample.items():
        value = str(value)

        if len(value) > 300:
            value = value[:300] + "..."

        print(f"{key}: {value}")

    print("\n")


inspect_dataset(farmerchat_raw, "FarmerChat Nigeria")
inspect_dataset(agronomy_raw, "Agronomy QA Pairs")

FarmerChat Nigeria

Number of records: 11541

Available columns:
- crop
- user_country
- user_geo_level2
- user_geo_level3
- query_year_month
- query
- response

Dataset features:
{'crop': Value('string'), 'user_country': Value('string'), 'user_geo_level2': Value('string'), 'user_geo_level3': Value('string'), 'query_year_month': Value('string'), 'query': Value('string'), 'response': Value('string')}

Sample record:
crop: generic
user_country: Nigeria
user_geo_level2: Kano
user_geo_level3: Kano Municipal
query_year_month: 2026-02
query: Should I water the leaves and the base of the plant or just the base when watering plants?
response: Hello there! It's great to hear from you. When it comes to watering your plants, it's generally best to focus on watering the base of the plant rather than the leaves. This is especially important in our region, where the climate can be quite hot and dry.

Watering the base ensures that the roots, w...


Agronomy QA Pairs

Number of records: 1914

Availab

## 5. Standardising Dataset Structures

FarmerChat Nigeria and Agronomy QA use different column names.

To make them compatible, we will map their fields into a shared
structure.

Important principles:
- Preserve the original question and answer text.
- Preserve source dataset information.
- Keep geographic and crop metadata where available.
- Leave unavailable metadata empty rather than guessing.
- Assign unique identifiers to every original record.

This step changes the structure, not the actual content.

In [7]:
# Convert the Hugging Face datasets into pandas DataFrames

farmerchat_df = farmerchat_raw.to_pandas()
agronomy_df = agronomy_raw.to_pandas()

# Standardise FarmerChat Nigeria
farmerchat_standard = pd.DataFrame({
    "record_id": [
        f"FCN_{i:05d}" for i in range(len(farmerchat_df))
    ],
    "source_dataset": "FarmerChat Nigeria",
    "source_record_id": farmerchat_df.index.astype(str),
    "question": farmerchat_df["query"],
    "answer": farmerchat_df["response"],
    "crop": farmerchat_df["crop"],
    "country": farmerchat_df["user_country"],
    "state_region": farmerchat_df["user_geo_level2"],
    "local_area": farmerchat_df["user_geo_level3"],
    "source_date": farmerchat_df["query_year_month"],
    "language": "English"
})

# Standardise Agronomy QA
agronomy_standard = pd.DataFrame({
    "record_id": [
        f"AQA_{i:05d}" for i in range(len(agronomy_df))
    ],
    "source_dataset": "Agronomy QA Pairs",
    "source_record_id": agronomy_df.index.astype(str),
    "question": agronomy_df["question"],
    "answer": agronomy_df["answer"],
    "crop": pd.NA,
    "country": pd.NA,
    "state_region": pd.NA,
    "local_area": pd.NA,
    "source_date": pd.NA,
    "language": "English"
})

print("Both datasets standardised successfully.")

Both datasets standardised successfully.


## 6. Combining the Standardised Datasets

We will combine the two standardised DataFrames into a single
working dataset.

The source_dataset and record_id fields will allow us to trace
each record back to its original dataset.

The original raw datasets remain untouched.

In [8]:
# Combine the standardised datasets

combined_df = pd.concat(
    [farmerchat_standard, agronomy_standard],
    ignore_index=True
)

print("Combined dataset created.")
print("Total records:", len(combined_df))

print("\nRecords by source:")
print(combined_df["source_dataset"].value_counts())

print("\nCombined dataset shape:")
print(combined_df.shape)

display(combined_df.head(5))

Combined dataset created.
Total records: 13455

Records by source:
source_dataset
FarmerChat Nigeria    11541
Agronomy QA Pairs      1914
Name: count, dtype: int64

Combined dataset shape:
(13455, 11)


,record_id,source_dataset,source_record_id,question,answer,crop,country,state_region,local_area,source_date,language
0,FCN_00000,FarmerChat Nigeria,0,Should I water the leaves and the base of the ...,Hello there! It's great to hear from you. When...,generic,Nigeria,Kano,Kano Municipal,2026-02,English
1,FCN_00001,FarmerChat Nigeria,1,When is the best time to apply fertilizer to m...,"Hello there! It's great to hear from you, and ...",generic,Nigeria,Edo,Etsako West,2026-02,English
2,FCN_00002,FarmerChat Nigeria,2,How can I protect my crops from the current we...,Hello there! 😊 It sounds like you're concerned...,generic,Nigeria,Rivers,Obio/Akpor,2026-02,English
3,FCN_00003,FarmerChat Nigeria,3,How does rainwater benefit plant growth during...,Hello! [XXX]. Let's talk about how rainwater b...,generic,Nigeria,Kano,Kano Municipal,2026-02,English
4,FCN_00004,FarmerChat Nigeria,4,What are the benefits of intercropping during ...,Hello there! 😊 Intercropping during the dry se...,generic,Nigeria,Rivers,Obio/Akpor,2026-02,English


## 7. Checking the Standardised Dataset

Before cleaning the text, we need to verify that the standardisation
process worked correctly.

We will check:
- Duplicate record identifiers
- Missing questions
- Missing answers
- Missing metadata
- Dataset structure

Missing metadata is not necessarily an error because the original
Agronomy QA dataset does not provide geographic or crop fields.
However, missing questions and answers require attention.

In [9]:
# Basic integrity checks

print("Duplicate record IDs:",
      combined_df["record_id"].duplicated().sum())

print("\nMissing values by column:")
print(combined_df.isnull().sum())

print("\nMissing question records:",
      combined_df["question"].isnull().sum())

print("\nMissing answer records:",
      combined_df["answer"].isnull().sum())

print("\nRecords by language:")
print(combined_df["language"].value_counts())

Duplicate record IDs: 0

Missing values by column:
record_id              0
source_dataset         0
source_record_id       0
question               0
answer                 0
crop                1914
country             1914
state_region        1914
local_area          1914
source_date         1914
language               0
dtype: int64

Missing question records: 0

Missing answer records: 0

Records by language:
language
English    13455
Name: count, dtype: int64


## 8. Cleaning Question and Answer Text

The original questions and answers may contain:
- Leading or trailing spaces
- Multiple consecutive spaces
- Newlines or tabs
- Inconsistent whitespace

We will standardise whitespace while preserving the original text.

Before cleaning, we will create backup columns so that the original
question and answer remain available for comparison.

We will not change spelling, grammar, punctuation, or agricultural
terminology at this stage.

In [10]:
# Preserve the original text before cleaning

combined_df["question_original"] = combined_df["question"]
combined_df["answer_original"] = combined_df["answer"]


def clean_text(value):
    """Remove unnecessary whitespace while preserving text meaning."""

    if pd.isna(value):
        return pd.NA

    text = str(value)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


# Clean questions and answers
combined_df["question"] = combined_df["question"].apply(clean_text)
combined_df["answer"] = combined_df["answer"].apply(clean_text)

print("Text cleaning completed.")
print("\nSample before and after cleaning:\n")

for index in range(min(3, len(combined_df))):
    print(f"Record: {combined_df.loc[index, 'record_id']}")
    print("Original question:", combined_df.loc[index, "question_original"])
    print("Cleaned question:", combined_df.loc[index, "question"])
    print()

Text cleaning completed.

Sample before and after cleaning:

Record: FCN_00000
Original question: Should I water the leaves and the base of the plant or just the base when watering plants?
Cleaned question: Should I water the leaves and the base of the plant or just the base when watering plants?

Record: FCN_00001
Original question: When is the best time to apply fertilizer to maize in Auchi, Edo, Nigeria?
Cleaned question: When is the best time to apply fertilizer to maize in Auchi, Edo, Nigeria?

Record: FCN_00002
Original question: How can I protect my crops from the current weather conditions in Obio/Akpor?
Cleaned question: How can I protect my crops from the current weather conditions in Obio/Akpor?



## 9. Checking Text Quality and Duplicate Records

Cleaning whitespace does not guarantee that all records are unique.

For example, the same question and answer may appear in both
datasets, or the same question may have different answers.

We will identify:
1. Empty questions after cleaning.
2. Empty answers after cleaning.
3. Exact duplicate questions.
4. Exact duplicate question-answer pairs.
5. Duplicate records within each source dataset.

For comparison, we will use lowercase text with standardised
whitespace. We will not remove punctuation because doing so could
make distinct agricultural questions appear identical.

No records will be deleted during this inspection.

In [11]:
# Check for empty text after cleaning

empty_questions = (
    combined_df["question"].isna() |
    combined_df["question"].eq("")
)

empty_answers = (
    combined_df["answer"].isna() |
    combined_df["answer"].eq("")
)

# Create comparison keys for duplicate detection
combined_df["question_key"] = (
    combined_df["question"].str.lower()
)

combined_df["answer_key"] = (
    combined_df["answer"].str.lower()
)

# Count duplicate questions and complete Q&A pairs
duplicate_questions = combined_df.duplicated(
    subset=["question_key"],
    keep=False
)

duplicate_qa_pairs = combined_df.duplicated(
    subset=["question_key", "answer_key"],
    keep=False
)

print("TEXT QUALITY")
print("-" * 40)
print("Empty questions:", empty_questions.sum())
print("Empty answers:", empty_answers.sum())

print("\nDUPLICATE ANALYSIS")
print("-" * 40)
print(
    "Records involved in repeated questions:",
    duplicate_questions.sum()
)
print(
    "Records involved in repeated Q&A pairs:",
    duplicate_qa_pairs.sum()
)

print("\nDUPLICATES BY SOURCE")
print("-" * 40)

duplicate_summary = combined_df.assign(
    repeated_question=duplicate_questions,
    repeated_qa_pair=duplicate_qa_pairs
).groupby("source_dataset").agg(
    repeated_questions=("repeated_question", "sum"),
    repeated_qa_pairs=("repeated_qa_pair", "sum")
)

display(duplicate_summary)

TEXT QUALITY
----------------------------------------
Empty questions: 0
Empty answers: 0

DUPLICATE ANALYSIS
----------------------------------------
Records involved in repeated questions: 4
Records involved in repeated Q&A pairs: 0

DUPLICATES BY SOURCE
----------------------------------------


,repeated_questions,repeated_qa_pairs
source_dataset,,
Agronomy QA Pairs,0,0
FarmerChat Nigeria,4,0


## 10. Investigating Repeated Questions

Some records contain the same question but different answers.

We need to inspect these records before making any deletion decisions.

For each repeated question, we will compare:
- Source dataset
- Answer
- Crop information
- Geographic metadata

Records will only be removed when there is a clear justification.
Different answers to the same question may provide useful alternative
information and should not automatically be discarded.

In [12]:
# Display all records involved in repeated questions

repeated_question_records = combined_df[
    duplicate_questions
].sort_values("question_key")

columns_to_inspect = [
    "record_id",
    "source_dataset",
    "question",
    "answer",
    "crop",
    "state_region",
    "local_area"
]

print("Repeated question records:")
display(repeated_question_records[columns_to_inspect])

print("\nDuplicate summary by source:")
print(duplicate_summary.to_string())

Repeated question records:


,record_id,source_dataset,question,answer,crop,state_region,local_area
6940,FCN_06940,FarmerChat Nigeria,What is the purpose of golvet injection for pi...,Hello there! It's great to hear from you again...,generic,Ekiti,Ado-Ekiti
6941,FCN_06941,FarmerChat Nigeria,What is the purpose of Golvet injection for pi...,Hello! It's great to continue our conversation...,generic,Ekiti,Ado-Ekiti
9651,FCN_09651,FarmerChat Nigeria,What issues might I be facing with my snake go...,Hello there! It sounds like you're having some...,generic,Akwa Ibom,Etim Ekpo
9652,FCN_09652,FarmerChat Nigeria,What issues might I be facing with my Snake Go...,Hello there! It's great to hear from you again...,generic,Akwa Ibom,Etim Ekpo



Duplicate summary by source:
                    repeated_questions  repeated_qa_pairs
source_dataset                                           
Agronomy QA Pairs                    0                  0
FarmerChat Nigeria                   4                  0


## 11. Recording Duplicate Review Information

The duplicate analysis identified repeated questions with different
answers.

Rather than deleting these records, we will assign a shared
repeated-question group identifier.

This allows us to:
- Track related records.
- Compare their answers later.
- Preserve source traceability.
- Make informed decisions during dataset selection.

No records will be removed at this stage.

In [13]:
# Identify groups of repeated questions

repeat_counts = combined_df["question_key"].value_counts()

repeated_keys = sorted(
    repeat_counts[repeat_counts > 1].index.tolist()
)

# Assign an identifier to each repeated-question group
repeat_group_lookup = {
    key: f"RQ_{i:03d}"
    for i, key in enumerate(repeated_keys, start=1)
}

combined_df["repeated_question_group"] = (
    combined_df["question_key"].map(repeat_group_lookup)
)

combined_df["is_repeated_question"] = (
    combined_df["repeated_question_group"].notna()
)

# Create an audit report
duplicate_review = combined_df[
    combined_df["is_repeated_question"]
][[
    "record_id",
    "source_dataset",
    "repeated_question_group",
    "question",
    "answer",
    "crop",
    "state_region",
    "local_area"
]].copy()

duplicate_review["review_status"] = "pending_manual_review"

# Save the report
duplicate_report_path = (
    METADATA_DIR / "repeated_question_review.csv"
)

duplicate_review.to_csv(
    duplicate_report_path,
    index=False
)

print("Duplicate review report saved:")
print(duplicate_report_path)

print("\nRepeated question groups:")
display(
    duplicate_review[
        ["record_id", "repeated_question_group",
         "question", "review_status"]
    ]
)

print("\nTotal repeated-question groups:", len(repeated_keys))
print("Total records involved:", len(duplicate_review))

Duplicate review report saved:
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata\repeated_question_review.csv

Repeated question groups:


,record_id,repeated_question_group,question,review_status
6940,FCN_06940,RQ_001,What is the purpose of golvet injection for pi...,pending_manual_review
6941,FCN_06941,RQ_001,What is the purpose of Golvet injection for pi...,pending_manual_review
9651,FCN_09651,RQ_002,What issues might I be facing with my snake go...,pending_manual_review
9652,FCN_09652,RQ_002,What issues might I be facing with my Snake Go...,pending_manual_review



Total repeated-question groups: 2
Total records involved: 4


## 12. Agricultural Relevance Filtering

EdoFarm AI initially focuses on seven agricultural areas:

1. Crop cultivation
2. Pest management
3. Disease management
4. Soil management
5. Irrigation
6. Fertilizer application
7. Harvesting

Livestock-related advisory questions are outside the initial scope.

We will use transparent keyword patterns to identify:
- Likely crop-related questions.
- Likely livestock-only questions.
- Questions requiring manual review.
- Possible agricultural intent categories.

Important:
- A question may match multiple categories.
- Keyword matches are preliminary, not verified labels.
- Questions will not be automatically deleted.
- Ambiguous records will be preserved for review.

In [14]:
# Define preliminary agricultural intent keywords

intent_patterns = {
    "crop_cultivation": [
        r"\bcultivat(?:e|ion|ing)\b",
        r"\bplant(?:ing|ed|s)?\b",
        r"\bgrow(?:ing|th|n)?\b",
        r"\bsow(?:ing|n)?\b",
        r"\bseed(?:s|ling)?\b",
        r"\btransplant(?:ing|ed)?\b",
        r"\bnursery\b",
        r"\bspacing\b",
        r"\bvariet(?:y|ies)\b",
        r"\bland preparation\b",
        r"\bpruning\b",
        r"\bpropagat(?:e|ion|ing)\b"
    ],

    "pest_management": [
        r"\bpests?\b",
        r"\binsects?\b",
        r"\baphids?\b",
        r"\bcaterpillars?\b",
        r"\bweevils?\b",
        r"\bstem borers?\b",
        r"\barmyworms?\b",
        r"\bwhiteflies\b",
        r"\bthrips\b",
        r"\bmites?\b",
        r"\btermites?\b",
        r"\blocusts?\b",
        r"\bgrasshoppers?\b",
        r"\bnematodes?\b"
    ],

    "disease_management": [
        r"\bdiseases?\b",
        r"\bfungal\b",
        r"\bfungus\b",
        r"\bbacterial\b",
        r"\bviral\b",
        r"\bvirus\b",
        r"\binfection\b",
        r"\binfected\b",
        r"\bblight\b",
        r"\bwilt\b",
        r"\brot\b",
        r"\bleaf spots?\b",
        r"\bmildew\b",
        r"\brust\b",
        r"\bmosaic\b",
        r"\banthracnose\b",
        r"\bdamping[- ]off\b",
        r"\bcanker\b"
    ],

    "soil_management": [
        r"\bsoil\b",
        r"\bsoil fertility\b",
        r"\bsoil erosion\b",
        r"\bsoil acidity\b",
        r"\bsoil texture\b",
        r"\bsoil drainage\b",
        r"\bsoil ph\b",
        r"\bph level\b",
        r"\bsalinity\b",
        r"\borganic matter\b",
        r"\bcompost\b",
        r"\bsoil nutrients?\b"
    ],

    "irrigation": [
        r"\birrigat(?:e|ion|ing)\b",
        r"\bwatering\b",
        r"\bwater supply\b",
        r"\brainfall\b",
        r"\bdrought\b",
        r"\bmoisture\b",
        r"\bdrip irrigation\b",
        r"\bsprinkler\b",
        r"\bwaterlogging\b",
        r"\bdry spell\b"
    ],

    "fertilizer_application": [
        r"\bfertili[sz](?:er|ers|ation|ing)\b",
        r"\bnpk\b",
        r"\burea\b",
        r"\bnitrogen\b",
        r"\bphosphorus\b",
        r"\bpotassium\b",
        r"\btop dressing\b",
        r"\bbasal application\b",
        r"\bfoliar feed(?:ing)?\b",
        r"\bfertilizer application\b"
    ],

    "harvesting": [
        r"\bharvest(?:ing)?\b",
        r"\bharvest maturity\b",
        r"\bripen(?:ing)?\b",
        r"\bripeness\b",
        r"\bpicking crops\b",
        r"\bpost[- ]harvest\b",
        r"\bcrop storage\b"
    ]
}


# Crop names relevant to Nigerian agriculture
crop_patterns = [
    r"\bcassava\b",
    r"\byam\b",
    r"\bmaize\b",
    r"\bcorn\b",
    r"\bplantain\b",
    r"\bbanana\b",
    r"\boil palm\b",
    r"\bpepper\b",
    r"\bokra\b",
    r"\bsnake gourd\b",
    r"\btomatoes?\b",
    r"\brice\b",
    r"\bcocoa\b",
    r"\bsorghum\b",
    r"\bmillet\b",
    r"\bgroundnuts?\b",
    r"\bcowpeas?\b",
    r"\bsoybeans?\b",
    r"\bcucumber\b",
    r"\beggplant\b",
    r"\bgarden eggs?\b",
    r"\bonions?\b",
    r"\bcabbage\b",
    r"\bsugarcane\b",
    r"\bcocoyam\b",
    r"\bsweet potato\b",
    r"\bpotatoes?\b",
    r"\boranges?\b",
    r"\bmango(?:es)?\b",
    r"\bpineapple\b",
    r"\bpawpaw\b",
    r"\bpapaya\b",
    r"\bwatermelon\b",
    r"\bcashew\b"
]


# General crop-production indicators
plant_agriculture_patterns = [
    r"\bcrops?\b",
    r"\bplants?\b",
    r"\bleaves\b",
    r"\bleaf\b",
    r"\broots?\b",
    r"\bstems?\b",
    r"\bvegetables?\b",
    r"\bfruits?\b",
    r"\bfarming\b",
    r"\bagriculture\b",
    r"\bseedlings?\b",
    r"\bplantation\b",
    r"\bfarmland\b"
]


# Livestock-related indicators
livestock_patterns = [
    r"\blivestock\b",
    r"\bpigs?\b",
    r"\bswine\b",
    r"\bpoultry\b",
    r"\bchickens?\b",
    r"\bbroilers?\b",
    r"\blayers?\b",
    r"\bcattle\b",
    r"\bcows?\b",
    r"\bcalves\b",
    r"\bgoats?\b",
    r"\bsheep\b",
    r"\bfish farming\b",
    r"\baquaculture\b",
    r"\banimals?\b",
    r"\bveterinary\b",
    r"\bvaccin(?:e|ation)\b",
    r"\bdeworm(?:ing)?\b",
    r"\binjections?\b"
]


print("Keyword patterns defined successfully.")
print("Agricultural intent categories:", len(intent_patterns))
print("Crop indicators:", len(crop_patterns))
print("Livestock indicators:", len(livestock_patterns))

Keyword patterns defined successfully.
Agricultural intent categories: 7
Crop indicators: 34
Livestock indicators: 19


## 13. Applying the Relevance Screening Rules

We will examine the question text and identify matching keywords.

Each record will receive:
- Matching agricultural intent categories.
- A crop-name match indicator.
- A general crop-production indicator.
- A livestock indicator.
- A preliminary relevance status.

The screening statuses are:

1. likely_in_scope
2. likely_out_of_scope
3. needs_review

A livestock question will only be considered likely out-of-scope
when no crop-production or agricultural intent indicators are found.

These statuses are screening suggestions, not final dataset labels.

In [15]:
# Compile keyword patterns for efficient matching

compiled_intents = {
    category: re.compile("|".join(patterns), re.IGNORECASE)
    for category, patterns in intent_patterns.items()
}

crop_regex = re.compile(
    "|".join(crop_patterns),
    re.IGNORECASE
)

plant_regex = re.compile(
    "|".join(plant_agriculture_patterns),
    re.IGNORECASE
)

livestock_regex = re.compile(
    "|".join(livestock_patterns),
    re.IGNORECASE
)


def screen_question(question):
    """Identify preliminary agricultural relevance signals."""

    text = str(question) if pd.notna(question) else ""

    matched_intents = [
        category
        for category, pattern in compiled_intents.items()
        if pattern.search(text)
    ]

    has_crop_mention = bool(crop_regex.search(text))
    has_plant_signal = bool(plant_regex.search(text))
    has_livestock_signal = bool(livestock_regex.search(text))

    has_agricultural_signal = (
        bool(matched_intents)
        or has_crop_mention
        or has_plant_signal
    )

    if has_agricultural_signal:
        status = "likely_in_scope"
    elif has_livestock_signal:
        status = "likely_out_of_scope"
    else:
        status = "needs_review"

    return pd.Series({
        "matched_intents": matched_intents,
        "has_crop_mention": has_crop_mention,
        "has_plant_signal": has_plant_signal,
        "has_livestock_signal": has_livestock_signal,
        "relevance_status": status
    })


# Apply screening to every record
screening_results = combined_df["question"].apply(screen_question)

combined_df = pd.concat(
    [combined_df, screening_results],
    axis=1
)

print("Agricultural relevance screening completed.")

Agricultural relevance screening completed.


## 14. Reviewing Agricultural Relevance Results

We will examine the overall screening distribution and the
preliminary intent-category coverage.

Intent categories can overlap. For example, a question about
applying compost to improve soil fertility may match both
soil management and fertilizer application.

Therefore, category counts should not be added together to
calculate unique agricultural records.

In [16]:
# Summary of relevance screening

print("RELEVANCE SCREENING SUMMARY")
print("-" * 45)

relevance_summary = combined_df["relevance_status"].value_counts()

display(relevance_summary.to_frame("record_count"))

print("\nPRELIMINARY INTENT COVERAGE")
print("-" * 45)

intent_summary = pd.Series({
    category: combined_df["matched_intents"].apply(
        lambda matches: category in matches
    ).sum()
    for category in intent_patterns
}, name="record_count")

display(intent_summary.to_frame())

print("\nCROP AND LIVESTOCK SIGNALS")
print("-" * 45)

signal_summary = combined_df[
    [
        "has_crop_mention",
        "has_plant_signal",
        "has_livestock_signal"
    ]
].sum()

display(signal_summary.to_frame("record_count"))

print("\nRecords needing manual review:",
      (combined_df["relevance_status"] == "needs_review").sum())

RELEVANCE SCREENING SUMMARY
---------------------------------------------


,record_count
relevance_status,
likely_in_scope,10920
needs_review,1413
likely_out_of_scope,1122



PRELIMINARY INTENT COVERAGE
---------------------------------------------


,record_count
crop_cultivation,3379
pest_management,1894
disease_management,1220
soil_management,394
irrigation,1058
fertilizer_application,1898
harvesting,348



CROP AND LIVESTOCK SIGNALS
---------------------------------------------


,record_count
has_crop_mention,6244
has_plant_signal,3786
has_livestock_signal,1538



Records needing manual review: 1413


## 15. Screening Distribution by Dataset

We will compare the relevance screening results across the two
original datasets.

This helps us identify whether one dataset contains significantly
more relevant records or whether the keyword rules behave
differently across sources.

We will also investigate why some records containing livestock
keywords were still classified as likely_in_scope.

In [17]:
# Compare screening results by original dataset

source_screening = pd.crosstab(
    combined_df["source_dataset"],
    combined_df["relevance_status"]
)

source_screening["total"] = source_screening.sum(axis=1)

print("SCREENING COUNTS BY SOURCE")
display(source_screening)

print("\nSCREENING PERCENTAGES BY SOURCE")

source_percentages = (
    pd.crosstab(
        combined_df["source_dataset"],
        combined_df["relevance_status"],
        normalize="index"
    ) * 100
)

display(source_percentages.round(2))

# Investigate records with livestock signals
print("\nLIVESTOCK SIGNAL OVERLAP")

livestock_summary = (
    combined_df[combined_df["has_livestock_signal"]]
    ["relevance_status"]
    .value_counts()
)

display(livestock_summary.to_frame("record_count"))

SCREENING COUNTS BY SOURCE


relevance_status,likely_in_scope,likely_out_of_scope,needs_review,total
source_dataset,,,,
Agronomy QA Pairs,1479,25,410,1914
FarmerChat Nigeria,9441,1097,1003,11541



SCREENING PERCENTAGES BY SOURCE


relevance_status,likely_in_scope,likely_out_of_scope,needs_review
source_dataset,,,
Agronomy QA Pairs,77.27,1.31,21.42
FarmerChat Nigeria,81.80,9.51,8.69



LIVESTOCK SIGNAL OVERLAP


,record_count
relevance_status,
likely_out_of_scope,1122
likely_in_scope,416


## 16. Manually Inspecting Screening Examples

Keyword-based filtering can produce false positives and false
negatives.

For example:
- A question about animal manure may be relevant to soil management.
- A question about livestock medication is outside the current scope.
- A question about crop disease may not contain the word "disease".
- A question about rainfall may concern irrigation or may be unrelated.

We will inspect representative examples from four groups:

1. Likely in-scope records that also contain livestock indicators.
2. Likely out-of-scope records.
3. Records needing review.
4. In-scope candidates identified through intent keywords without
   explicit crop or plant indicators.

This is a preliminary manual audit, not a substitute for expert
agricultural review.

In [18]:
# Columns useful for reviewing agricultural relevance

review_columns = [
    "record_id",
    "source_dataset",
    "question",
    "crop",
    "state_region",
    "local_area",
    "matched_intents",
    "has_crop_mention",
    "has_plant_signal",
    "has_livestock_signal",
    "relevance_status"
]


def inspect_sample(dataframe, title, sample_size=10):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)

    if dataframe.empty:
        print("No records found in this group.")
        return

    sample = dataframe.sample(
        n=min(sample_size, len(dataframe)),
        random_state=42
    )

    display(sample[review_columns])


# 1. Agricultural and livestock indicators occur together
inspect_sample(
    combined_df[
        (combined_df["has_livestock_signal"]) &
        (combined_df["relevance_status"] == "likely_in_scope")
    ],
    "IN-SCOPE CANDIDATES WITH LIVESTOCK SIGNALS"
)

# 2. Likely out-of-scope records
inspect_sample(
    combined_df[
        combined_df["relevance_status"] == "likely_out_of_scope"
    ],
    "LIKELY OUT-OF-SCOPE RECORDS"
)

# 3. Records requiring review
inspect_sample(
    combined_df[
        combined_df["relevance_status"] == "needs_review"
    ],
    "RECORDS NEEDING REVIEW"
)

# 4. Records classified through intent keywords alone
inspect_sample(
    combined_df[
        (combined_df["relevance_status"] == "likely_in_scope") &
        (~combined_df["has_crop_mention"]) &
        (~combined_df["has_plant_signal"])
    ],
    "IN-SCOPE CANDIDATES WITHOUT EXPLICIT CROP OR PLANT SIGNALS"
)


IN-SCOPE CANDIDATES WITH LIVESTOCK SIGNALS


,record_id,source_dataset,question,crop,state_region,local_area,matched_intents,has_crop_mention,has_plant_signal,has_livestock_signal,relevance_status
4732,FCN_04732,FarmerChat Nigeria,When is the best time to start poultry farming...,generic,Delta,Ndokwa West,[],False,True,True,likely_in_scope
8070,FCN_08070,FarmerChat Nigeria,How should I plan the feeding schedule for my ...,generic,Imo,Isu,[crop_cultivation],False,False,True,likely_in_scope
11855,AQA_00314,Agronomy QA Pairs,name any livestock diseases.,<NA>,<NA>,<NA>,[disease_management],False,False,True,likely_in_scope
2824,FCN_02824,FarmerChat Nigeria,Which week are you referring to in the context...,generic,Federal Capital Territory,Abuja Municipal Area Council,[],False,True,True,likely_in_scope
2157,FCN_02157,FarmerChat Nigeria,What are the best practices for pig farming in...,generic,Ogun State,Ado Odo/Ota,[],False,True,True,likely_in_scope
11016,FCN_11016,FarmerChat Nigeria,"Can goats consume groundnut hay, and is it a s...",generic,Oyo,Ibadan North East,[],True,False,True,likely_in_scope
9752,FCN_09752,FarmerChat Nigeria,What are the best practices for general care i...,generic,Bayelsa,Ogbia,[],False,True,True,likely_in_scope
6826,FCN_06826,FarmerChat Nigeria,How can I effectively use pig manure as fertil...,generic,Ekiti,Ado-Ekiti,[fertilizer_application],True,False,True,likely_in_scope
940,FCN_00940,FarmerChat Nigeria,What is the feed conversion ratio for pigs and...,generic,Delta,Warri South,[crop_cultivation],False,False,True,likely_in_scope
1548,FCN_01548,FarmerChat Nigeria,"I need assistance with an agricultural issue, ...",generic,Zamfara,Kaura Namoda,[],False,True,True,likely_in_scope



LIKELY OUT-OF-SCOPE RECORDS


,record_id,source_dataset,question,crop,state_region,local_area,matched_intents,has_crop_mention,has_plant_signal,has_livestock_signal,relevance_status
9853,FCN_09853,FarmerChat Nigeria,What nutrition should adult goats receive now ...,generic,Oyo,Ibadan North East,[],False,False,True,likely_out_of_scope
8678,FCN_08678,FarmerChat Nigeria,How does the local weather in Eti-Osa affect t...,generic,Lagos,Eti Osa,[],False,False,True,likely_out_of_scope
744,FCN_00744,FarmerChat Nigeria,What will be the weight of a broiler chicken a...,generic,Kano,Tofa,[],False,False,True,likely_out_of_scope
7656,FCN_07656,FarmerChat Nigeria,How should I adjust the feeding routine for my...,generic,Federal Capital Territory,Abuja Municipal Area Council,[],False,False,True,likely_out_of_scope
10993,FCN_10993,FarmerChat Nigeria,What does lethargy in chickens indicate and wh...,generic,Kano,Bagwai,[],False,False,True,likely_out_of_scope
8679,FCN_08679,FarmerChat Nigeria,What feeding strategies can help my broiler ch...,generic,Lagos,Eti Osa,[],False,False,True,likely_out_of_scope
4396,FCN_04396,FarmerChat Nigeria,How can I improve the feed quality for my shee...,generic,Katsina,Mani,[],False,False,True,likely_out_of_scope
11321,FCN_11321,FarmerChat Nigeria,What are the best feed options for goats durin...,generic,Oyo,Lagelu,[],False,False,True,likely_out_of_scope
2050,FCN_02050,FarmerChat Nigeria,What are the benefits of providing salt licks ...,generic,Ogun State,Odeda,[],False,False,True,likely_out_of_scope
7049,FCN_07049,FarmerChat Nigeria,What are the best practices for nursing local ...,generic,Oyo,Oyo West,[],False,False,True,likely_out_of_scope



RECORDS NEEDING REVIEW


,record_id,source_dataset,question,crop,state_region,local_area,matched_intents,has_crop_mention,has_plant_signal,has_livestock_signal,relevance_status
8756,FCN_08756,FarmerChat Nigeria,What marketing strategies can help increase my...,generic,Niger,Bida,[],False,False,False,needs_review
9909,FCN_09909,FarmerChat Nigeria,What are the best practices for brooding manag...,generic,Bayelsa,Ogbia,[],False,False,False,needs_review
3842,FCN_03842,FarmerChat Nigeria,What are the effects of overfeeding in young p...,generic,Ekiti,Ado-Ekiti,[],False,False,False,needs_review
2284,FCN_02284,FarmerChat Nigeria,What are the best management practices for gui...,generic,Katsina,Mai'Adua,[],False,False,False,needs_review
2437,FCN_02437,FarmerChat Nigeria,What are the best practices to prevent wheat l...,generic,Rivers,Obio/Akpor,[],False,False,False,needs_review
8504,FCN_08504,FarmerChat Nigeria,What are the solutions for water problems in E...,generic,Lagos,Eti Osa,[],False,False,False,needs_review
5936,FCN_05936,FarmerChat Nigeria,What methods can I use to prevent water stress...,generic,Kaduna,Lere,[],False,False,False,needs_review
908,FCN_00908,FarmerChat Nigeria,What practices can improve pollination for pea...,generic,Oyo,Ido,[],False,False,False,needs_review
11290,FCN_11290,FarmerChat Nigeria,How many months can a duck carry pregnancy in ...,generic,Oyo,Ibadan North East,[],False,False,False,needs_review
648,FCN_00648,FarmerChat Nigeria,I would like to know more about agricultural o...,generic,Plateau,Jos North,[],False,False,False,needs_review



IN-SCOPE CANDIDATES WITHOUT EXPLICIT CROP OR PLANT SIGNALS


,record_id,source_dataset,question,crop,state_region,local_area,matched_intents,has_crop_mention,has_plant_signal,has_livestock_signal,relevance_status
12275,AQA_00734,Agronomy QA Pairs,Why is diammonium phosphate put in planting hole.,<NA>,<NA>,<NA>,[crop_cultivation],False,False,False,likely_in_scope
328,FCN_00328,FarmerChat Nigeria,How should I store my beans after harvest to a...,generic,Kano,Nassarawa,"[pest_management, harvesting]",False,False,False,likely_in_scope
13384,AQA_01843,Agronomy QA Pairs,What are common pests and diseases affecting g...,<NA>,<NA>,<NA>,"[pest_management, disease_management]",False,False,False,likely_in_scope
7441,FCN_07441,FarmerChat Nigeria,"How is cadmon planted in Zaria City, Kwarbai A...",generic,Kaduna,Zaria,[crop_cultivation],False,False,False,likely_in_scope
9269,FCN_09269,FarmerChat Nigeria,How can I manage pests in mature bananas in Ak...,generic,Oyo,Lagelu,[pest_management],False,False,False,likely_in_scope
4644,FCN_04644,FarmerChat Nigeria,What natural methods can I use to manage pests...,generic,Plateau,Bokkos,"[crop_cultivation, pest_management]",False,False,False,likely_in_scope
976,FCN_00976,FarmerChat Nigeria,What practices can be implemented to prevent n...,generic,Lagos,Alimosho,[crop_cultivation],False,False,False,likely_in_scope
5412,FCN_05412,FarmerChat Nigeria,How should I adjust my watering schedule given...,generic,Kano,Bichi,[irrigation],False,False,False,likely_in_scope
5596,FCN_05596,FarmerChat Nigeria,What are the best pest control methods for leg...,generic,Federal Capital Territory,Abuja Municipal Area Council,[pest_management],False,False,False,likely_in_scope
4733,FCN_04733,FarmerChat Nigeria,How can I effectively manage pests in my poult...,generic,Delta,Ndokwa West,[pest_management],False,False,True,likely_in_scope


## 17. Refining Agricultural Relevance Rules

The first screening process revealed several classification problems.

The revised rules will address:

1. Livestock-only questions being incorrectly classified as crop-related.
2. Crop-related uses of livestock by-products, such as manure, being
   incorrectly excluded.
3. Missing crop names and agricultural terminology.
4. Agricultural topics such as pollination, lodging and water stress
   being overlooked.
5. Agricultural business and marketing questions being distinguished
   from crop-production advisory.

The original screening results will be preserved.

A second screening version will be generated for comparison.

The resulting classifications remain preliminary and require
manual review before records are selected for the final dataset.

In [19]:
# Create an improved copy of our agricultural intent dictionary.
# The original intent_patterns dictionary remains unchanged.

intent_patterns_v2 = {
    category: patterns.copy()
    for category, patterns in intent_patterns.items()
}

# Add missing crop-production terminology
intent_patterns_v2["crop_cultivation"] += [
    r"\bpollinat(?:e|ion|ing)\b",
    r"\blodging\b",
    r"\bintercropping\b",
    r"\bweeding\b",
    r"\bgermination\b",
    r"\bplanting hole\b",
    r"\bseed rate\b",
    r"\bplant population\b",
    r"\bcrop rotation\b",
    r"\bmulching\b"
]

# Improve soil management coverage
intent_patterns_v2["soil_management"] += [
    r"\bmanure\b",
    r"\bdung\b",
    r"\bcompost(?:ing)?\b",
    r"\bsoil amendment\b",
    r"\borganic matter\b"
]

# Improve irrigation terminology
intent_patterns_v2["irrigation"] += [
    r"\bwater stress\b",
    r"\bmoisture stress\b",
    r"\bwater shortage\b",
    r"\bwater scarcity\b",
    r"\bwater supply problems?\b"
]

# Improve fertilizer terminology
intent_patterns_v2["fertilizer_application"] += [
    r"\bdap\b",
    r"\bdiammonium phosphate\b",
    r"\banimal manure\b",
    r"\bpoultry manure\b",
    r"\bfarmyard manure\b",
    r"\borganic manure\b"
]

# Improve harvesting and storage terminology
intent_patterns_v2["harvesting"] += [
    r"\bstor(?:e|ing|ed)\b",
    r"\bstorage conditions\b",
    r"\bpostharvest\b",
    r"\bpost harvest\b"
]


# Recompile the updated intent patterns
compiled_intents_v2 = {
    category: re.compile(
        "|".join(patterns),
        re.IGNORECASE
    )
    for category, patterns in intent_patterns_v2.items()
}


# Add crop names that were missing from the original dictionary
additional_crop_patterns = [
    r"\bbeans?\b",
    r"\bpeas?\b",
    r"\bwheat\b",
    r"\bguava\b",
    r"\bginger\b",
    r"\bgarlic\b",
    r"\bspinach\b",
    r"\blettuce\b",
    r"\bwaterleaf\b",
    r"\bcitrus\b",
    r"\bguinea corn\b",
    r"\blegumes?\b",
    r"\bmelon\b",
    r"\bbitter leaf\b",
    r"\bfluted pumpkin\b"
]

crop_regex_v2 = re.compile(
    "|".join(crop_patterns + additional_crop_patterns),
    re.IGNORECASE
)


# Remove overly broad indicators such as "farming" and "agriculture".
# These words alone do not establish crop-production relevance.

crop_context_patterns_v2 = [
    r"\bcrops?\b",
    r"\bplants?\b",
    r"\bseedlings?\b",
    r"\bplanting\b",
    r"\bsowing\b",
    r"\bcultivat\w*\b",
    r"\bharvest\w*\b",
    r"\bintercropping\b",
    r"\bweeding\b",
    r"\bpollinat\w*\b",
    r"\bsoil\b",
    r"\bcompost\b",
    r"\bfertilizer\b",
    r"\bfertili[sz]\w*\b",
    r"\birrigat\w*\b",
    r"\bwater stress\b",
    r"\bplant nutrition\b",
    r"\bcrop rotation\b",
    r"\bplanting hole\b"
]

crop_context_regex_v2 = re.compile(
    "|".join(crop_context_patterns_v2),
    re.IGNORECASE
)


# More specific livestock indicators
livestock_patterns_v2 = [
    r"\blivestock\b",
    r"\bpoultry\b",
    r"\bchickens?\b",
    r"\bchicks?\b",
    r"\bbroilers?\b",
    r"\blayers?\b",
    r"\bguinea fowl\b",
    r"\bducks?\b",
    r"\bturkeys?\b",
    r"\bgeese\b",
    r"\bpigs?\b",
    r"\bpiglets?\b",
    r"\bswine\b",
    r"\bgoats?\b",
    r"\bsheep\b",
    r"\blambs?\b",
    r"\bcattle\b",
    r"\bcows?\b",
    r"\bcalves\b",
    r"\bfish farming\b",
    r"\baquaculture\b",
    r"\bbrooding\b",
    r"\banimal husbandry\b",
    r"\banimal nutrition\b",
    r"\banimal feeding\b",
    r"\bfeeding schedule\b",
    r"\bfeeding routine\b",
    r"\bfeed conversion ratio\b",
    r"\banimal health\b",
    r"\banimal care\b",
    r"\banimal diseases?\b",
    r"\bveterinary\b",
    r"\bvaccine\b",
    r"\bvaccination\b",
    r"\bdeworming\b",
    r"\bmilk production\b",
    r"\begg production\b",
    r"\bmeat production\b"
]

livestock_regex_v2 = re.compile(
    "|".join(livestock_patterns_v2),
    re.IGNORECASE
)


# Specific crop-production uses of animal by-products.
# These can be relevant even when livestock terminology appears.

manure_regex_v2 = re.compile(
    r"\b(?:manure|dung|animal waste|poultry litter)\b",
    re.IGNORECASE
)

crop_use_regex_v2 = re.compile(
    r"\b(?:fertili[sz]\w*|compost\w*|soil|crop|crops|plants|"
    r"maize|cassava|yam|garden|field|vegetables?)\b",
    re.IGNORECASE
)


# Topics outside the current crop-production advisory scope
non_crop_patterns_v2 = [
    r"\bmarketing strategies?\b",
    r"\bmarket prices?\b",
    r"\bfind buyers?\b",
    r"\bsell(?:ing)? my produce\b",
    r"\bfarm loans?\b",
    r"\bagricultural grants?\b",
    r"\bsubsid(?:y|ies)\b",
    r"\bprofit margins?\b",
    r"\bbusiness plan\b"
]

non_crop_regex_v2 = re.compile(
    "|".join(non_crop_patterns_v2),
    re.IGNORECASE
)

print("Refined screening dictionaries created.")
print("Intent categories:", len(compiled_intents_v2))
print("Livestock patterns:", len(livestock_patterns_v2))
print("Additional crop patterns:", len(additional_crop_patterns))

Refined screening dictionaries created.
Intent categories: 7
Livestock patterns: 38
Additional crop patterns: 15


## 18. Applying the Revised Relevance Screening

The second screening process uses more specific decision rules.

Decision order:

1. Livestock-related questions are classified as likely_out_of_scope,
   unless they specifically concern using animal by-products for
   crop production.

2. Explicit agricultural business and marketing questions without
   a crop-production intent are classified as likely_out_of_scope.

3. Questions matching an agricultural intent, crop name, or
   crop-production context are classified as likely_in_scope.

4. Questions without sufficient evidence remain needs_review.

These classifications are preliminary screening decisions.
They are not final human-verified agricultural labels.

In [20]:
def screen_question_v2(question):
    """Apply the refined crop-production relevance rules."""

    text = str(question) if pd.notna(question) else ""

    # Identify agricultural intents
    matched_intents = [
        category
        for category, pattern in compiled_intents_v2.items()
        if pattern.search(text)
    ]

    # Identify crop and crop-production signals
    has_crop_mention = bool(crop_regex_v2.search(text))

    has_crop_context = bool(
        crop_context_regex_v2.search(text)
    )

    # Identify livestock-related content
    has_livestock_signal = bool(
        livestock_regex_v2.search(text)
    )

    # Animal by-products used in crop production
    has_manure_signal = bool(
        manure_regex_v2.search(text)
    )

    has_crop_use_signal = bool(
        crop_use_regex_v2.search(text)
    )

    has_crop_input_use = (
        has_manure_signal and has_crop_use_signal
    )

    # Identify non-crop business topics
    has_non_crop_topic = bool(
        non_crop_regex_v2.search(text)
    )

    # Determine preliminary relevance
    if has_livestock_signal and not has_crop_input_use:
        status = "likely_out_of_scope"

    elif (
        has_non_crop_topic
        and not matched_intents
        and not has_crop_input_use
    ):
        status = "likely_out_of_scope"

    elif (
        matched_intents
        or has_crop_mention
        or has_crop_context
        or has_crop_input_use
    ):
        status = "likely_in_scope"

    else:
        status = "needs_review"

    return pd.Series({
        "matched_intents_v2": matched_intents,
        "has_crop_mention_v2": has_crop_mention,
        "has_crop_context_v2": has_crop_context,
        "has_livestock_signal_v2": has_livestock_signal,
        "has_crop_input_use_v2": has_crop_input_use,
        "has_non_crop_topic_v2": has_non_crop_topic,
        "relevance_status_v2": status
    })


# Apply the revised screening to the combined dataset
screening_results_v2 = combined_df["question"].apply(
    screen_question_v2
)

combined_df = pd.concat(
    [combined_df, screening_results_v2],
    axis=1
)

print("Revised agricultural relevance screening completed.")

Revised agricultural relevance screening completed.


## 19. Comparing the Original and Revised Screening

We will compare the first screening version with the revised version.

This will help us understand:
- How many records changed classification.
- How many livestock-related records are now excluded.
- Whether records involving manure and crop production remain eligible.
- Whether the revised rules produce a more reasonable distribution.

Neither version is treated as ground truth.

In [21]:
print("ORIGINAL SCREENING")
display(
    combined_df["relevance_status"]
    .value_counts()
    .to_frame("record_count")
)

print("\nREVISED SCREENING")
display(
    combined_df["relevance_status_v2"]
    .value_counts()
    .to_frame("record_count")
)

print("\nSOURCE DISTRIBUTION: REVISED SCREENING")

display(
    pd.crosstab(
        combined_df["source_dataset"],
        combined_df["relevance_status_v2"],
        margins=True
    )
)

print("\nCLASSIFICATION CHANGES")

comparison = pd.crosstab(
    combined_df["relevance_status"],
    combined_df["relevance_status_v2"],
    margins=True
)

display(comparison)

changed_records = combined_df[
    combined_df["relevance_status"] !=
    combined_df["relevance_status_v2"]
]

print("\nRecords whose classification changed:",
      len(changed_records))

ORIGINAL SCREENING


,record_count
relevance_status,
likely_in_scope,10920
needs_review,1413
likely_out_of_scope,1122



REVISED SCREENING


,record_count
relevance_status_v2,
likely_in_scope,10435
likely_out_of_scope,1694
needs_review,1326



SOURCE DISTRIBUTION: REVISED SCREENING


relevance_status_v2,likely_in_scope,likely_out_of_scope,needs_review,All
source_dataset,,,,
Agronomy QA Pairs,1515,32,367,1914
FarmerChat Nigeria,8920,1662,959,11541
All,10435,1694,1326,13455



CLASSIFICATION CHANGES


relevance_status_v2,likely_in_scope,likely_out_of_scope,needs_review,All
relevance_status,,,,
likely_in_scope,10125,462,333,10920
likely_out_of_scope,2,1103,17,1122
needs_review,308,129,976,1413
All,10435,1694,1326,13455



Records whose classification changed: 1251


## 20. Auditing Changes Between Screening Versions

The revised screening changed the classification of 1,251 records.

We will inspect representative examples from the following transitions:

1. likely_in_scope → likely_out_of_scope
2. likely_in_scope → needs_review
3. needs_review → likely_in_scope
4. needs_review → likely_out_of_scope
5. likely_out_of_scope → likely_in_scope
6. likely_out_of_scope → needs_review

This audit will help identify false positives and false negatives
introduced by the revised screening rules.

The original and revised classifications will both be preserved.

In [22]:
# Define the classification transitions to investigate

transitions = [
    ("likely_in_scope", "likely_out_of_scope"),
    ("likely_in_scope", "needs_review"),
    ("needs_review", "likely_in_scope"),
    ("needs_review", "likely_out_of_scope"),
    ("likely_out_of_scope", "likely_in_scope"),
    ("likely_out_of_scope", "needs_review")
]

audit_columns = [
    "record_id",
    "source_dataset",
    "question",
    "answer",
    "matched_intents",
    "matched_intents_v2",
    "has_crop_mention",
    "has_livestock_signal",
    "has_crop_input_use_v2",
    "relevance_status",
    "relevance_status_v2"
]


for old_status, new_status in transitions:

    group = combined_df[
        (combined_df["relevance_status"] == old_status) &
        (combined_df["relevance_status_v2"] == new_status)
    ]

    print("\n" + "=" * 85)
    print(f"{old_status} → {new_status}")
    print("Total records:", len(group))
    print("=" * 85)

    if not group.empty:
        sample = group.sample(
            n=min(8, len(group)),
            random_state=42
        )

        display(sample[audit_columns])


likely_in_scope → likely_out_of_scope
Total records: 462


,record_id,source_dataset,question,answer,matched_intents,matched_intents_v2,has_crop_mention,has_livestock_signal,has_crop_input_use_v2,relevance_status,relevance_status_v2
6628,FCN_06628,FarmerChat Nigeria,When should I vaccinate my broilers to prevent...,"Hello! [XXX], and I'm glad you're focusing on ...",[disease_management],[disease_management],False,True,False,likely_in_scope,likely_out_of_scope
671,FCN_00671,FarmerChat Nigeria,Will I achieve a weight of 2.5 kilograms for m...,Hello! [XXX]. Achieving a weight of 2.5 kilogr...,[],[],True,True,False,likely_in_scope,likely_out_of_scope
827,FCN_00827,FarmerChat Nigeria,What are the signs of heartwater disease in sh...,Hello! [XXX]. Let's talk about the signs of he...,[disease_management],[disease_management],False,True,False,likely_in_scope,likely_out_of_scope
6520,FCN_06520,FarmerChat Nigeria,What is the feed formula for broiler chickens ...,Hello there! It's great to hear from a farmer ...,[],[],True,True,False,likely_in_scope,likely_out_of_scope
3528,FCN_03528,FarmerChat Nigeria,What are the benefits and considerations of fe...,Hello there! It's wonderful to hear from you a...,[],[],True,False,False,likely_in_scope,likely_out_of_scope
5829,FCN_05829,FarmerChat Nigeria,What are the best treatment options for ticks ...,Hello there! It's great to hear from you. Tick...,[disease_management],[disease_management],False,True,False,likely_in_scope,likely_out_of_scope
9727,FCN_09727,FarmerChat Nigeria,Can molasses be mixed with crushed cassava pee...,Hello there! It's great to hear from you. Mixi...,[],[],True,True,False,likely_in_scope,likely_out_of_scope
6247,FCN_06247,FarmerChat Nigeria,How should I plan the feeding schedule for my ...,Hello! It's great to see your commitment to ra...,[crop_cultivation],[crop_cultivation],False,False,False,likely_in_scope,likely_out_of_scope



likely_in_scope → needs_review
Total records: 333


,record_id,source_dataset,question,answer,matched_intents,matched_intents_v2,has_crop_mention,has_livestock_signal,has_crop_input_use_v2,relevance_status,relevance_status_v2
1334,FCN_01334,FarmerChat Nigeria,How can I access local grants for tomato farmi...,"Hello! [XXX], and I'm glad you're interested i...",[],[],False,False,False,likely_in_scope,needs_review
12477,AQA_00936,Agronomy QA Pairs,Explain the use of AI/ML & Data Science in Agr...,Utilizes artificial intelligence and machine l...,[],[],False,False,False,likely_in_scope,needs_review
2851,FCN_02851,FarmerChat Nigeria,What does latillaged mean in the context of fa...,Hello there! It's great to hear from you. It s...,[],[],False,False,False,likely_in_scope,needs_review
7978,FCN_07978,FarmerChat Nigeria,How can I prepare for foggy conditions in my f...,Hello there! It's great to hear from you. Prep...,[],[],False,False,False,likely_in_scope,needs_review
2469,FCN_02469,FarmerChat Nigeria,"What are the causes of browning, yellowing, an...","Hello there! It's great to hear from you, and ...",[],[],False,False,False,likely_in_scope,needs_review
12457,AQA_00916,Agronomy QA Pairs,Describe how the Pradhanmantri Krishi Sinchaye...,This policy focuses on enhancing water use eff...,[],[],False,False,False,likely_in_scope,needs_review
11868,AQA_00327,Agronomy QA Pairs,how does climate change affects agriculture.,"through changes in average temperatures, rainf...",[],[],False,False,False,likely_in_scope,needs_review
7251,FCN_07251,FarmerChat Nigeria,What is the role of liquid soap as a foliar ap...,Hello there! It's wonderful to see your intere...,[],[],False,False,False,likely_in_scope,needs_review



needs_review → likely_in_scope
Total records: 308


,record_id,source_dataset,question,answer,matched_intents,matched_intents_v2,has_crop_mention,has_livestock_signal,has_crop_input_use_v2,relevance_status,relevance_status_v2
12977,AQA_01436,Agronomy QA Pairs,What are the recommended practices for storing...,"Apples should be stored in a cool, humid envir...",[],[harvesting],False,False,False,needs_review,likely_in_scope
931,FCN_00931,FarmerChat Nigeria,What nutrients should I apply to improve wheat...,"Hello there! It's great to hear from you, and ...",[],[],False,False,False,needs_review,likely_in_scope
4153,FCN_04153,FarmerChat Nigeria,What are the best chemicals for beans in Oko E...,Hello there! It's wonderful to hear from you a...,[],[],False,False,False,needs_review,likely_in_scope
4161,FCN_04161,FarmerChat Nigeria,What is the expected yield of wheat with natur...,Hello [XXX]! It's great to continue our discus...,[],[],False,False,False,needs_review,likely_in_scope
1967,FCN_01967,FarmerChat Nigeria,How can I manage water stress during coffee fl...,"Hello there! It's great to hear from you, and ...",[],[irrigation],False,False,False,needs_review,likely_in_scope
4205,FCN_04205,FarmerChat Nigeria,Is it appropriate to farm ginger in Lere LGA?,Hello! [XXX]. You're asking about the suitabil...,[],[],False,False,False,needs_review,likely_in_scope
5566,FCN_05566,FarmerChat Nigeria,What steps should I take to prepare my legume ...,Hello! Preparing your legume fields for the up...,[],[],False,False,False,needs_review,likely_in_scope
10910,FCN_10910,FarmerChat Nigeria,How can I manage heat stress during the flower...,"Hello there! It's great to hear from you, and ...",[],[],False,False,False,needs_review,likely_in_scope


... (2 outputs left out)


likely_out_of_scope → likely_in_scope
Total records: 2


,record_id,source_dataset,question,answer,matched_intents,matched_intents_v2,has_crop_mention,has_livestock_signal,has_crop_input_use_v2,relevance_status,relevance_status_v2
3747,FCN_03747,FarmerChat Nigeria,What is the recommended ratio of sawdust to an...,Hello there! It's great to see your enthusiasm...,[],"[soil_management, fertilizer_application]",False,True,False,likely_out_of_scope,likely_in_scope
3746,FCN_03746,FarmerChat Nigeria,What is the recommended ratio for mixing anima...,Hello [XXX]! It's wonderful to see your contin...,[],"[soil_management, fertilizer_application]",False,True,False,likely_out_of_scope,likely_in_scope



likely_out_of_scope → needs_review
Total records: 17


,record_id,source_dataset,question,answer,matched_intents,matched_intents_v2,has_crop_mention,has_livestock_signal,has_crop_input_use_v2,relevance_status,relevance_status_v2
486,FCN_00486,FarmerChat Nigeria,What kinds of animals and birds can directly e...,Hello! It's great to see your interest in usin...,[],[],False,True,False,likely_out_of_scope,needs_review
956,FCN_00956,FarmerChat Nigeria,How can I encourage my animals to drink water ...,Hello there! It's great to hear from you. Ensu...,[],[],False,True,False,likely_out_of_scope,needs_review
4012,FCN_04012,FarmerChat Nigeria,What kind of protection is appropriate to prot...,Hello there! It's wonderful to connect with yo...,[],[],False,True,False,likely_out_of_scope,needs_review
11677,AQA_00136,Agronomy QA Pairs,what is animal welfare?,refers to the conditions in which an animal is...,[],[],False,True,False,likely_out_of_scope,needs_review
9547,FCN_09547,FarmerChat Nigeria,What is the name of the injection used for hat...,"Hello! [XXX]. When it comes to breeding fish, ...",[],[],False,True,False,likely_out_of_scope,needs_review
11672,AQA_00131,Agronomy QA Pairs,why has Food and Drug Administration has appro...,"to treat disease in animals that are sick, con...",[],[],False,True,False,likely_out_of_scope,needs_review
5258,FCN_05258,FarmerChat Nigeria,What type of feed should I provide my animals ...,Hello there! It's great to hear from a farmer ...,[],[],False,True,False,likely_out_of_scope,needs_review
11055,FCN_11055,FarmerChat Nigeria,What is the importance of water for the health...,Hello [XXX]! It's wonderful to continue our co...,[],[],False,True,False,likely_out_of_scope,needs_review


## 21. Inspecting Revised Screening Signals

The previous audit displayed some original screening indicators
alongside the revised classifications.

We will now inspect the actual Version 2 indicators.

For each example, we will examine:
- Full question text.
- Original and revised classification.
- Revised crop-name detection.
- Revised crop-production context detection.
- Revised livestock detection.
- Revised manure-use detection.
- Revised matched intent categories.

We will also inspect specific records where the screening
rules may need adjustment.

No records will be deleted or relabelled in this step.

In [24]:
# Inspect transitions using the correct Version 2 indicators

audit_columns_v2 = [
    "record_id",
    "source_dataset",
    "question",
    "relevance_status",
    "relevance_status_v2",
    "matched_intents_v2",
    "has_crop_mention_v2",
    "has_crop_context_v2",
    "has_livestock_signal_v2",
    "has_crop_input_use_v2"
]

transitions_to_inspect = [
    ("likely_in_scope", "likely_out_of_scope"),
    ("likely_in_scope", "needs_review"),
    ("needs_review", "likely_in_scope"),
    ("needs_review", "likely_out_of_scope"),
    ("likely_out_of_scope", "likely_in_scope"),
    ("likely_out_of_scope", "needs_review")
]

for old_status, new_status in transitions_to_inspect:

    group = combined_df[
        (combined_df["relevance_status"] == old_status) &
        (combined_df["relevance_status_v2"] == new_status)
    ]

    print("\n" + "=" * 100)
    print(f"{old_status} -> {new_status}")
    print("Total records:", len(group))
    print("=" * 100)

    if not group.empty:
        sample = group.sample(
            n=min(4, len(group)),
            random_state=42
        )

        print(
            sample[audit_columns_v2].to_string(
                index=False,
                max_colwidth=None
            )
        )


# Examine specific records that deserve closer attention
target_ids = [
    "FCN_03746",
    "FCN_03747",
    "FCN_06247",
    "FCN_03528",
    "FCN_00486",
    "AQA_00136",
    "FCN_09547"
]

print("\n" + "=" * 100)
print("TARGETED RECORD INSPECTION")
print("=" * 100)

target_records = combined_df[
    combined_df["record_id"].isin(target_ids)
]

print(
    target_records[audit_columns_v2].to_string(
        index=False,
        max_colwidth=None
    )
)


likely_in_scope -> likely_out_of_scope
Total records: 462
record_id     source_dataset                                                                                                                                    question relevance_status relevance_status_v2   matched_intents_v2  has_crop_mention_v2  has_crop_context_v2  has_livestock_signal_v2  has_crop_input_use_v2
FCN_06628 FarmerChat Nigeria                                                                  When should I vaccinate my broilers to prevent common diseases in Igbuzor?  likely_in_scope likely_out_of_scope [disease_management]                False                False                     True                  False
FCN_00671 FarmerChat Nigeria Will I achieve a weight of 2.5 kilograms for my broilers in 6 weeks with the current feed mixture of guinea corn, millet, and starter feed?  likely_in_scope likely_out_of_scope                   []                 True                False                     True              

## 22. Final Refinement of Agricultural Relevance Screening

The audit identified several remaining limitations in Version 2.

Version 3 introduces the following corrections:

1. Animal manure must have an identifiable crop-production use
   before it qualifies as a crop input.
2. Animal welfare and aquaculture questions are explicitly excluded.
3. Mushroom substrate and mushroom cultivation questions are flagged
   for manual review because their scope has not yet been established.
4. Agricultural grants, market prices, marketing and general
   agricultural technology questions are excluded unless they also
   contain a genuine crop-production advisory intent.
5. A crop name appearing by itself is not sufficient to establish
   agricultural relevance.

Version 2 will remain unchanged for comparison.

Version 3 still represents preliminary screening, not expert-verified
classification.

In [25]:
# Create an independent copy of Version 2's intent patterns

intent_patterns_v3 = {
    category: patterns.copy()
    for category, patterns in intent_patterns_v2.items()
}

# Remove generic manure terms from agricultural intent categories.
# Manure must be evaluated in its actual crop-production context.

generic_manure_patterns = {
    r"\bmanure\b",
    r"\bdung\b",
    r"\banimal manure\b",
    r"\bpoultry manure\b",
    r"\bfarmyard manure\b",
    r"\borganic manure\b"
}

for category in ["soil_management", "fertilizer_application"]:
    intent_patterns_v3[category] = [
        pattern
        for pattern in intent_patterns_v3[category]
        if pattern not in generic_manure_patterns
    ]

# Add more specific crop-production terminology

intent_patterns_v3["crop_cultivation"] += [
    r"\bcrop yield\b",
    r"\byield of\b",
    r"\bcrop productivity\b",
    r"\bgrowing conditions\b",
    r"\bsuitability of growing\b",
    r"\bflowering stage\b"
]

intent_patterns_v3["fertilizer_application"] += [
    r"\bnutrients?\b",
    r"\bplant nutrition\b",
    r"\bfoliar application\b",
    r"\bfoliar spray\b"
]

# Compile Version 3 intent patterns

compiled_intents_v3 = {
    category: re.compile(
        "|".join(patterns),
        re.IGNORECASE
    )
    for category, patterns in intent_patterns_v3.items()
}


# Additional animal and aquaculture terminology

livestock_patterns_v3 = livestock_patterns_v2 + [
    r"\banimal welfare\b",
    r"\bfish hatchery\b",
    r"\bhatching fish\b",
    r"\bfish hatching\b",
    r"\bfish breeding\b",
    r"\bfingerlings?\b",
    r"\bfish eggs?\b",
    r"\bfish ponds?\b",
    r"\baquaculture\b",
    r"\bfishery\b",
    r"\bfisheries\b"
]

livestock_regex_v3 = re.compile(
    "|".join(livestock_patterns_v3),
    re.IGNORECASE
)


# Identify animal manure and related agricultural by-products

manure_regex_v3 = re.compile(
    r"\b(?:manure|dung|animal waste|poultry litter)\b",
    re.IGNORECASE
)

crop_input_context_v3 = re.compile(
    r"\b(?:soil|crop|crops|maize|cassava|yam|plantain|"
    r"pepper|okra|vegetables?|fertili[sz]\w*|"
    r"soil fertility|crop production|"
    r"apply|application|incorporat\w*)\b",
    re.IGNORECASE
)


# Mushroom cultivation and substrate questions require manual review

mushroom_regex_v3 = re.compile(
    r"\b(?:mushrooms?|mycelium|mycelia|mushroom spawn|"
    r"mushroom substrate)\b",
    re.IGNORECASE
)


# Agricultural business and general technology topics

non_crop_patterns_v3 = [
    r"\bmarketing strategies?\b",
    r"\bmarket prices?\b",
    r"\bfind buyers?\b",
    r"\bsell(?:ing)? my produce\b",
    r"\bfarm loans?\b",
    r"\bagricultural grants?\b",
    r"\blocal grants?\b",
    r"\bsubsid(?:y|ies)\b",
    r"\bprofit margins?\b",
    r"\bbusiness plan\b",
    r"\bai/?ml\b",
    r"\bdata science\b",
    r"\bmachine learning\b"
]

non_crop_regex_v3 = re.compile(
    "|".join(non_crop_patterns_v3),
    re.IGNORECASE
)

print("Version 3 screening rules created.")
print("Intent categories:", len(compiled_intents_v3))
print("Livestock patterns:", len(livestock_patterns_v3))

Version 3 screening rules created.
Intent categories: 7
Livestock patterns: 49


## 23. Applying Version 3 Screening Rules

Version 3 uses stricter decision conditions.

Classification order:

1. Mushroom-related questions are sent for manual review.
2. Livestock and aquaculture questions are out-of-scope unless
   they explicitly concern agricultural crop-input use.
3. Agricultural business and general technology questions are
   out-of-scope when they do not contain a crop-production intent.
4. Questions with identifiable crop-production intents are
   likely_in_scope.
5. Other ambiguous questions remain needs_review.

A crop name alone cannot automatically qualify a question.

We will create a separate Version 3 audit table so that the
original datasets and earlier screening versions remain preserved.

In [26]:
def screen_question_v3(question):
    """Apply Version 3 crop-production relevance rules."""

    text = str(question) if pd.notna(question) else ""

    # Identify agricultural intent categories
    matched_intents = [
        category
        for category, pattern in compiled_intents_v3.items()
        if pattern.search(text)
    ]

    # Identify relevant signals
    has_livestock = bool(livestock_regex_v3.search(text))
    has_mushroom = bool(mushroom_regex_v3.search(text))

    has_manure = bool(manure_regex_v3.search(text))

    has_crop_input_use = (
        has_manure and
        bool(crop_input_context_v3.search(text))
    )

    has_non_crop_topic = bool(non_crop_regex_v3.search(text))

    # Assign preliminary screening status

    if has_mushroom:
        status = "needs_review"

    elif has_livestock and not has_crop_input_use:
        status = "likely_out_of_scope"

    elif (
        has_non_crop_topic
        and not matched_intents
        and not has_crop_input_use
    ):
        status = "likely_out_of_scope"

    elif has_crop_input_use:
        status = "likely_in_scope"

        if "fertilizer_application" not in matched_intents:
            matched_intents.append("fertilizer_application")

    elif matched_intents:
        status = "likely_in_scope"

    else:
        status = "needs_review"

    return {
        "matched_intents_v3": matched_intents,
        "has_livestock_signal_v3": has_livestock,
        "has_mushroom_signal_v3": has_mushroom,
        "has_crop_input_use_v3": has_crop_input_use,
        "relevance_status_v3": status
    }


# Create a clean, independent audit table

base_columns = [
    "record_id",
    "source_dataset",
    "question",
    "answer"
]

audit_v3 = combined_df[base_columns].copy().reset_index(drop=True)

# Retrieve the latest Version 2 status if multiple copies exist
v2_column = combined_df.loc[
    :, combined_df.columns == "relevance_status_v2"
]

if isinstance(v2_column, pd.DataFrame):
    audit_v3["relevance_status_v2"] = (
        v2_column.iloc[:, -1].reset_index(drop=True)
    )
else:
    audit_v3["relevance_status_v2"] = (
        v2_column.reset_index(drop=True)
    )

# Apply Version 3
v3_results = audit_v3["question"].apply(
    screen_question_v3
).apply(pd.Series)

audit_v3 = pd.concat(
    [audit_v3, v3_results.reset_index(drop=True)],
    axis=1
)

print("Version 3 screening completed.")
print("Total records:", len(audit_v3))

print("\nVERSION 3 DISTRIBUTION")
display(
    audit_v3["relevance_status_v3"]
    .value_counts()
    .to_frame("record_count")
)

print("\nVERSION 2 VS VERSION 3")

display(
    pd.crosstab(
        audit_v3["relevance_status_v2"],
        audit_v3["relevance_status_v3"],
        margins=True
    )
)

print("\nSOURCE DISTRIBUTION")

display(
    pd.crosstab(
        audit_v3["source_dataset"],
        audit_v3["relevance_status_v3"],
        margins=True
    )
)

# Inspect important previously identified records

target_ids = [
    "FCN_03746",
    "FCN_03747",
    "FCN_06247",
    "FCN_09547",
    "AQA_00136",
    "FCN_01334",
    "AQA_00936"
]

print("\nTARGETED RECORD REVIEW")

display(
    audit_v3[
        audit_v3["record_id"].isin(target_ids)
    ][[
        "record_id",
        "question",
        "matched_intents_v3",
        "has_livestock_signal_v3",
        "has_mushroom_signal_v3",
        "has_crop_input_use_v3",
        "relevance_status_v3"
    ]]
)

Version 3 screening completed.
Total records: 13455

VERSION 3 DISTRIBUTION


,record_count
relevance_status_v3,
likely_in_scope,8951
needs_review,2783
likely_out_of_scope,1721



VERSION 2 VS VERSION 3


relevance_status_v3,likely_in_scope,likely_out_of_scope,needs_review,All
relevance_status_v2,,,,
likely_in_scope,8916,6,1513,10435
likely_out_of_scope,0,1693,1,1694
needs_review,35,22,1269,1326
All,8951,1721,2783,13455



SOURCE DISTRIBUTION


relevance_status_v3,likely_in_scope,likely_out_of_scope,needs_review,All
source_dataset,,,,
Agronomy QA Pairs,1245,35,634,1914
FarmerChat Nigeria,7706,1686,2149,11541
All,8951,1721,2783,13455



TARGETED RECORD REVIEW


,record_id,question,matched_intents_v3,has_livestock_signal_v3,has_mushroom_signal_v3,has_crop_input_use_v3,relevance_status_v3
1334,FCN_01334,How can I access local grants for tomato farmi...,[],False,False,False,likely_out_of_scope
3746,FCN_03746,What is the recommended ratio for mixing anima...,[],False,True,False,needs_review
3747,FCN_03747,What is the recommended ratio of sawdust to an...,[],False,True,False,needs_review
6247,FCN_06247,How should I plan the feeding schedule for my ...,[crop_cultivation],True,False,False,likely_out_of_scope
9547,FCN_09547,What is the name of the injection used for hat...,[],True,False,False,likely_out_of_scope
11677,AQA_00136,what is animal welfare?,[],True,False,False,likely_out_of_scope
12477,AQA_00936,Explain the use of AI/ML & Data Science in Agr...,[],False,False,False,likely_out_of_scope


## 24. Agricultural Intent Coverage Analysis

We will analyse the 8,951 likely in-scope candidates from Version 3.

The objectives are to:
- Count records associated with each agricultural intent.
- Compare category coverage across both source datasets.
- Identify categories with relatively few examples.
- Determine how many questions match one or multiple categories.

The keyword-matched categories are preliminary suggestions.
They are not final training labels.

Each selected question will eventually receive a manually
reviewed primary intent label.

In [27]:
# Extract likely in-scope candidates
in_scope_candidates = audit_v3[
    audit_v3["relevance_status_v3"] == "likely_in_scope"
].copy()

# Count the number of matched intents per record
in_scope_candidates["intent_count"] = (
    in_scope_candidates["matched_intents_v3"]
    .apply(lambda x: len(x) if isinstance(x, list) else 0)
)

print("LIKELY IN-SCOPE CANDIDATES")
print("-" * 45)
print("Total records:", len(in_scope_candidates))
print(
    "Unique record IDs:",
    in_scope_candidates["record_id"].nunique()
)

# Convert the intent lists into individual rows
intent_rows = in_scope_candidates.explode(
    "matched_intents_v3"
)

intent_rows = intent_rows.dropna(
    subset=["matched_intents_v3"]
)

# Count unique records per category
category_summary = (
    intent_rows.groupby("matched_intents_v3")
    .agg(
        unique_records=("record_id", "nunique")
    )
    .sort_values("unique_records", ascending=False)
)

print("\nRECORDS PER AGRICULTURAL CATEGORY")
display(category_summary)

# Compare the two sources
source_category_summary = pd.crosstab(
    intent_rows["matched_intents_v3"],
    intent_rows["source_dataset"]
)

source_category_summary["total"] = (
    source_category_summary.sum(axis=1)
)

print("\nCATEGORY COVERAGE BY SOURCE")
display(source_category_summary)

# Examine single-intent and multiple-intent records
print("\nNUMBER OF MATCHED INTENTS PER RECORD")
display(
    in_scope_candidates["intent_count"]
    .value_counts()
    .sort_index()
    .to_frame("record_count")
)

# Check whether any in-scope candidates lack an intent match
unmatched_in_scope = in_scope_candidates[
    in_scope_candidates["intent_count"] == 0
]

print(
    "\nIn-scope records without a matched intent:",
    len(unmatched_in_scope)
)

LIKELY IN-SCOPE CANDIDATES
---------------------------------------------
Total records: 8951
Unique record IDs: 8951

RECORDS PER AGRICULTURAL CATEGORY


,unique_records
matched_intents_v3,
crop_cultivation,4324
fertilizer_application,2081
pest_management,1869
irrigation,1122
disease_management,1086
harvesting,405
soil_management,389



CATEGORY COVERAGE BY SOURCE


source_dataset,Agronomy QA Pairs,FarmerChat Nigeria,total
matched_intents_v3,,,
crop_cultivation,556,3768,4324
disease_management,489,597,1086
fertilizer_application,124,1957,2081
harvesting,97,308,405
irrigation,69,1053,1122
pest_management,119,1750,1869
soil_management,133,256,389



NUMBER OF MATCHED INTENTS PER RECORD


,record_count
intent_count,
1,6689
2,2200
3,61
4,1



In-scope records without a matched intent: 0


## 25. Building the Manual Review Candidate Pool

The purpose of this stage is to create a manageable collection
of records for human quality assessment.

Our proposed final dataset contains 352 examples:
44 for each of the eight classification categories.

To allow for rejection of unsuitable examples, we will initially
sample up to 66 candidates per category:

- 44 from FarmerChat Nigeria.
- 22 from Agronomy QA Pairs.

This gives priority to Nigerian agricultural content while
retaining useful additional agricultural knowledge.

A record that matches multiple categories may be selected
for more than one category during sampling.

We will combine repeated selections by record_id so that
every underlying record appears only once in the review queue.

The candidate pool is not yet the final training dataset.
All selected questions and answers must undergo manual review.

In [28]:
# Use the independent Version 3 audit table.
# This avoids any duplicate-column issues in combined_df.

categories = [
    "crop_cultivation",
    "pest_management",
    "disease_management",
    "soil_management",
    "irrigation",
    "fertilizer_application",
    "harvesting"
]

# Proposed sampling quotas
source_quotas = {
    "FarmerChat Nigeria": 44,
    "Agronomy QA Pairs": 22
}

random_seed = 42

# Verify the audit table
assert audit_v3["record_id"].is_unique, (
    "Duplicate record IDs found in the Version 3 audit table."
)

# Attach original geographic and crop metadata from FarmerChat.
metadata_columns = [
    "record_id",
    "crop",
    "country",
    "state_region",
    "local_area",
    "source_date"
]

farmerchat_metadata = (
    farmerchat_standard[metadata_columns]
    .drop_duplicates(subset=["record_id"])
)

candidate_base = audit_v3.merge(
    farmerchat_metadata,
    on="record_id",
    how="left",
    validate="one_to_one"
)

# Check for repeated questions without deleting them
candidate_base["question_key"] = (
    candidate_base["question"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

question_counts = candidate_base[
    "question_key"
].value_counts()

candidate_base["exact_question_duplicate_count"] = (
    candidate_base["question_key"]
    .map(question_counts)
)

# Function to identify records matching an intent
def contains_intent(intent_list, category):
    return (
        isinstance(intent_list, (list, tuple, set))
        and category in intent_list
    )


# Create the candidate selections
sampled_groups = []

for category in categories + ["out_of_scope"]:

    if category == "out_of_scope":
        category_pool = candidate_base[
            candidate_base["relevance_status_v3"]
            == "likely_out_of_scope"
        ].copy()

    else:
        category_pool = candidate_base[
            (
                candidate_base["relevance_status_v3"]
                == "likely_in_scope"
            )
            &
            candidate_base["matched_intents_v3"].apply(
                lambda intents: contains_intent(intents, category)
            )
        ].copy()

    for source, quota in source_quotas.items():

        source_pool = category_pool[
            category_pool["source_dataset"] == source
        ]

        sample_size = min(quota, len(source_pool))

        selected = source_pool.sample(
            n=sample_size,
            random_state=random_seed
        ).copy()

        selected["review_candidate_for"] = category

        sampled_groups.append(selected)

        print(
            f"{category:25} | "
            f"{source:22} | "
            f"{sample_size:3} selected"
        )


# Combine all candidate selections
raw_review_pool = pd.concat(
    sampled_groups,
    ignore_index=True
)

print("\nTotal candidate selections:", len(raw_review_pool))

crop_cultivation          | FarmerChat Nigeria     |  44 selected
crop_cultivation          | Agronomy QA Pairs      |  22 selected
pest_management           | FarmerChat Nigeria     |  44 selected
pest_management           | Agronomy QA Pairs      |  22 selected
disease_management        | FarmerChat Nigeria     |  44 selected
disease_management        | Agronomy QA Pairs      |  22 selected
soil_management           | FarmerChat Nigeria     |  44 selected
soil_management           | Agronomy QA Pairs      |  22 selected
irrigation                | FarmerChat Nigeria     |  44 selected
irrigation                | Agronomy QA Pairs      |  22 selected
fertilizer_application    | FarmerChat Nigeria     |  44 selected
fertilizer_application    | Agronomy QA Pairs      |  22 selected
harvesting                | FarmerChat Nigeria     |  44 selected
harvesting                | Agronomy QA Pairs      |  22 selected
out_of_scope              | FarmerChat Nigeria     |  44 selected
out_of_sco

## 26. Combining the Review Candidates

A record may appear in more than one agricultural category
because its question can cover multiple topics.

For example, a question about applying fertilizer to improve
soil fertility may match both fertilizer application and
soil management.

We will combine all selections using the unique record_id.

Each record will retain:
- Its original question and answer.
- Its source dataset.
- Its available geographic and crop metadata.
- All categories for which it was selected.
- A preliminary suggested primary intent.

Records selected for multiple categories will require manual
primary-intent assignment.

In [30]:


source_record_lookup = pd.concat(
    [
        farmerchat_standard[["record_id", "source_record_id"]],
        agronomy_standard[["record_id", "source_record_id"]]
    ],
    ignore_index=True
).drop_duplicates(subset=["record_id"])

# Add the missing field to the candidate pool
if "source_record_id" not in raw_review_pool.columns:
    raw_review_pool = raw_review_pool.merge(
        source_record_lookup,
        on="record_id",
        how="left",
        validate="many_to_one"
    )

# Verify source traceability
print(
    "Missing source record IDs:",
    raw_review_pool["source_record_id"].isna().sum()
)

# Combine repeated candidate selections into unique records
review_queue = (
    raw_review_pool
    .groupby("record_id", as_index=False)
    .agg(
        source_dataset=("source_dataset", "first"),
        source_record_id=("source_record_id", "first"),
        question=("question", "first"),
        answer=("answer", "first"),
        crop=("crop", "first"),
        country=("country", "first"),
        state_region=("state_region", "first"),
        local_area=("local_area", "first"),
        source_date=("source_date", "first"),
        matched_intents_v3=("matched_intents_v3", "first"),
        relevance_status_v3=("relevance_status_v3", "first"),
        exact_question_duplicate_count=(
            "exact_question_duplicate_count", "max"
        ),
        candidate_intents=(
            "review_candidate_for",
            lambda values: sorted(set(values))
        )
    )
)

# Suggest a primary intent only when there is one candidate category
review_queue["proposed_primary_intent"] = (
    review_queue["candidate_intents"].apply(
        lambda values: values[0] if len(values) == 1 else ""
    )
)

# Create empty fields for human review
review_queue["primary_intent"] = ""
review_queue["review_decision"] = ""
review_queue["answer_quality"] = ""
review_queue["agricultural_accuracy"] = ""
review_queue["nigerian_relevance"] = ""
review_queue["review_notes"] = ""
review_queue["review_status"] = "pending"

print("\nManual review queue created.")

print("Raw candidate selections:", len(raw_review_pool))
print("Unique records:", len(review_queue))

print("\nReview queue by source:")
display(review_queue["source_dataset"].value_counts().to_frame("record_count"))

print("\nReview queue by candidate category:")

category_review_summary = pd.Series({
    category: review_queue["candidate_intents"].apply(
        lambda values: category in values
    ).sum()
    for category in categories + ["out_of_scope"]
}, name="unique_review_records")

display(category_review_summary.to_frame())

print(
    "\nRecords requiring primary-intent assignment:",
    (review_queue["proposed_primary_intent"] == "").sum()
)

print(
    "\nRecords with repeated questions:",
    (review_queue["exact_question_duplicate_count"] > 1).sum()
)

Missing source record IDs: 0

Manual review queue created.
Raw candidate selections: 528
Unique records: 522

Review queue by source:


,record_count
source_dataset,
FarmerChat Nigeria,348
Agronomy QA Pairs,174



Review queue by candidate category:


,unique_review_records
crop_cultivation,66
pest_management,66
disease_management,66
soil_management,66
irrigation,66
fertilizer_application,66
harvesting,66
out_of_scope,66



Records requiring primary-intent assignment: 6

Records with repeated questions: 0


## 27. Exporting the Manual Review Queue

The review queue will be exported as a CSV file.

It will be stored in the metadata directory, separately from
the original datasets and processed agricultural records.

The review columns are intentionally blank because the
agricultural quality assessments have not yet been performed.

This file can also be shared with team members so that
multiple reviewers can assess different records.

In [31]:
# Save the review queue

review_queue_path = (
    METADATA_DIR / "manual_review_queue_v1.csv"
)

review_queue.to_csv(
    review_queue_path,
    index=False
)

print("Manual review queue saved successfully.")
print(review_queue_path)

print("\nExported records:", len(review_queue))
print("Exported columns:", len(review_queue.columns))

Manual review queue saved successfully.
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata\manual_review_queue_v1.csv

Exported records: 522
Exported columns: 22


## 28. Manual Review and Quality Assurance

The initial candidate pool contains 522 unique records.

Each record will undergo manual assessment before inclusion
in the final dataset.

The assessment covers:

1. Agricultural relevance.
2. Correct primary intent classification.
3. Answer quality and completeness.
4. Agricultural accuracy and safety.
5. Relevance to Nigerian agricultural conditions.
6. Duplicate or near-duplicate content.

Records may be accepted, rejected, or flagged for expert review.

Out-of-scope questions will be retained as classification
examples where appropriate. Their original answers will not
automatically be treated as approved agricultural advice.

The final dataset will contain 352 unique underlying examples,
with 44 examples assigned to each of the eight classification
categories.

English and Edo translations will share the same underlying
example identifier and will not count as separate examples.

Manual review by the project team does not constitute
professional agricultural or native-speaker validation.

## 29. Manual Review Without Agricultural Expert Validation

No agricultural expert is currently available to review the dataset.

The project team will therefore conduct preliminary quality
assessment, focusing on:

1. Relevance to the chatbot's intended scope.
2. Clarity and understandability of questions.
3. Correct assignment of agricultural intent categories.
4. Completeness and readability of answers.
5. Identification of potentially unsafe or misleading advice.
6. General relevance to Nigerian agricultural conditions.

The team will not claim to verify scientific agricultural accuracy.

All records will carry the status:
not_expert_verified

Questions and labels accepted by the team may be used for
intent-classification development.

Answers will not automatically be approved for model fine-tuning
or inclusion in the retrieval knowledge base.

Agricultural knowledge used for retrieval must undergo separate
source verification.

In [32]:
# Create a new review copy.
# The original Version 1 CSV remains unchanged.

review_queue_v2 = review_queue.copy()

# Identification and review tracking
review_queue_v2["reviewer_id"] = ""
review_queue_v2["review_date"] = ""

# Team review fields
review_queue_v2["review_decision"] = "pending"
review_queue_v2["primary_intent"] = ""
review_queue_v2["answer_quality"] = "not_assessed"
review_queue_v2["nigerian_relevance"] = "not_assessed"
review_queue_v2["review_notes"] = ""

# Agricultural accuracy and expert-validation tracking
review_queue_v2["agricultural_accuracy"] = "not_expert_verified"
review_queue_v2["expert_review_status"] = "not_available"

# Safety and source verification
review_queue_v2["safety_flags"] = ""
review_queue_v2["source_check_status"] = "not_checked"

# Intended use of each record
review_queue_v2["classifier_label_status"] = "pending"
review_queue_v2["knowledge_base_status"] = (
    "not_approved_pending_source_verification"
)

review_queue_v2["review_status"] = "pending"

# Save a new version
review_queue_v2_path = (
    METADATA_DIR / "manual_review_queue_v2.csv"
)

review_queue_v2.to_csv(
    review_queue_v2_path,
    index=False
)

print("Version 2 review queue created.")
print("Total records:", len(review_queue_v2))
print("Unique record IDs:", review_queue_v2["record_id"].nunique())
print("Agricultural expert validation: Not available")
print("\nSaved to:")
print(review_queue_v2_path)

display(
    review_queue_v2[
        [
            "record_id",
            "question",
            "candidate_intents",
            "review_decision",
            "primary_intent",
            "agricultural_accuracy",
            "expert_review_status"
        ]
    ].head(5)
)

Version 2 review queue created.
Total records: 522
Unique record IDs: 522
Agricultural expert validation: Not available

Saved to:
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata\manual_review_queue_v2.csv


,record_id,question,candidate_intents,review_decision,primary_intent,agricultural_accuracy,expert_review_status
0,AQA_00002,what are the different methods of irrigation?,[irrigation],pending,,not_expert_verified,not_available
1,AQA_00009,which diseases can be spread by whiteflies?,[pest_management],pending,,not_expert_verified,not_available
2,AQA_00027,why is Integrated Pest Management used?,[pest_management],pending,,not_expert_verified,not_available
3,AQA_00031,What is the best way to store harvested wheat ...,[harvesting],pending,,not_expert_verified,not_available
4,AQA_00044,What type of soil is best suited for growing b...,[soil_management],pending,,not_expert_verified,not_available


## 30. Manual Review Guidelines

Each candidate record must be assessed before inclusion
in the final dataset.

### A. Review decision

Use one of:

- accept
- reject
- needs_second_review

### B. Primary intent

Choose exactly one:

- crop_cultivation
- pest_management
- disease_management
- soil_management
- irrigation
- fertilizer_application
- harvesting
- out_of_scope

The final label must reflect the main purpose of the question,
not every keyword it contains.

### C. Answer quality

Use one of:

- clear_and_relevant
- unclear_or_incomplete
- potentially_misleading
- unusable
- not_assessed

These describe preliminary answer quality, not scientific validity.

### D. Nigerian relevance

Use one of:

- directly_nigerian
- generally_transferable
- foreign_context_specific
- unclear
- not_assessed

### E. Safety flags

Record any concerns, including:

- Unsupported pesticide recommendations.
- Unspecified or unverified chemical dosages.
- Definitive disease diagnoses without sufficient evidence.
- Unsafe application instructions.
- Unsupported guarantees about crop yields.
- Advice that may be unsuitable for Nigerian conditions.

Do not assume that the absence of a safety flag means
an answer is scientifically correct.

### F. Acceptance rules

Accept a record for intent-classification development when
its question is understandable, relevant to its assigned
category, and has a defensible primary label.

Reject records that are incoherent, unsuitable, unusable,
or unrelated to the intended classification task.

Send ambiguous records for a second review.

A record accepted by the project team is not automatically
agriculturally expert-verified or approved for the knowledge base.

In [33]:
# Inspect the review queue before team members begin reviewing

print("REVIEW QUEUE SUMMARY")
print("-" * 45)

print("Total records:", len(review_queue_v2))

print(
    "\nRecords by source:"
)

display(
    review_queue_v2["source_dataset"]
    .value_counts()
    .to_frame("record_count")
)

print("\nRecords by candidate category:")

category_summary_v2 = pd.Series({
    category: review_queue_v2["candidate_intents"].apply(
        lambda values: category in values
    ).sum()
    for category in categories + ["out_of_scope"]
}, name="candidate_count")

display(category_summary_v2.to_frame())

print("\nReview status:")
display(
    review_queue_v2["review_status"]
    .value_counts()
    .to_frame("record_count")
)

print("\nExpert validation status:")
display(
    review_queue_v2["agricultural_accuracy"]
    .value_counts()
    .to_frame("record_count")
)

print("\nReview file:")
print(review_queue_v2_path)

REVIEW QUEUE SUMMARY
---------------------------------------------
Total records: 522

Records by source:


,record_count
source_dataset,
FarmerChat Nigeria,348
Agronomy QA Pairs,174



Records by candidate category:


,candidate_count
crop_cultivation,66
pest_management,66
disease_management,66
soil_management,66
irrigation,66
fertilizer_application,66
harvesting,66
out_of_scope,66



Review status:


,record_count
review_status,
pending,522



Expert validation status:


,record_count
agricultural_accuracy,
not_expert_verified,522



Review file:
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata\manual_review_queue_v2.csv


## 31. Final Dataset Preparation and Export

In this stage, we use an AI-assisted review as the working assessment for the EdoFarm AI prototype.

The original dataset remains unchanged.

We will create separate datasets for:

1. Intent classifier training candidates.
2. Rejected records.
3. Records requiring second review.
4. Agricultural knowledge candidates.
5. Review statistics.

An important distinction is maintained between classifier suitability and answer quality. A question may be useful for intent classification even if its original answer is incomplete or unsuitable for agricultural knowledge retrieval.

All classifier labels and agricultural knowledge candidates remain provisional pending appropriate confirmation and verification.

In [34]:
from pathlib import Path
import pandas as pd

# Locate the project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# File paths
original_path = METADATA_DIR / "manual_review_queue_v2.csv"
reviewed_path = METADATA_DIR / "edofarm_full_reviewed_dataset.csv"
sidecar_path = METADATA_DIR / "edofarm_full_review_sidecar.csv"

# Check that all three files exist
for path in [original_path, reviewed_path, sidecar_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            "Place the second AI's output files in data/metadata "
            "using the expected filenames."
        )

# Load without changing missing values or converting IDs
original_df = pd.read_csv(
    original_path,
    dtype=str,
    keep_default_na=False
)

reviewed_df = pd.read_csv(
    reviewed_path,
    dtype=str,
    keep_default_na=False
)

sidecar_df = pd.read_csv(
    sidecar_path,
    dtype=str,
    keep_default_na=False
)

print("Files loaded successfully.")
print("Original dataset:", original_df.shape)
print("Reviewed dataset:", reviewed_df.shape)
print("Review sidecar:", sidecar_df.shape)

Files loaded successfully.
Original dataset: (522, 29)
Reviewed dataset: (522, 29)
Review sidecar: (522, 16)


### 31.1 Structural Integrity Verification

Before extracting any training or knowledge-base candidates, verify that:

- The reviewed dataset retains the original 29-column schema.
- No original records have been lost or added.
- Record IDs remain in their original order.
- Original questions and answers remain unchanged.
- Original source and preliminary-screening fields remain unchanged.
- The sidecar aligns with the corresponding review fields in the reviewed dataset.

No filtering or record removal is performed during this check.

In [35]:
EXPECTED_COLUMNS = [
    "record_id",
    "source_dataset",
    "source_record_id",
    "question",
    "answer",
    "crop",
    "country",
    "state_region",
    "local_area",
    "source_date",
    "matched_intents_v3",
    "relevance_status_v3",
    "exact_question_duplicate_count",
    "candidate_intents",
    "proposed_primary_intent",
    "primary_intent",
    "review_decision",
    "answer_quality",
    "agricultural_accuracy",
    "nigerian_relevance",
    "review_notes",
    "review_status",
    "reviewer_id",
    "review_date",
    "expert_review_status",
    "safety_flags",
    "source_check_status",
    "classifier_label_status",
    "knowledge_base_status"
]

assert list(original_df.columns) == EXPECTED_COLUMNS
assert list(reviewed_df.columns) == EXPECTED_COLUMNS
assert reviewed_df.shape[1] == 29

# Verify record IDs and order
assert original_df["record_id"].equals(reviewed_df["record_id"])
assert original_df["record_id"].is_unique
assert reviewed_df["record_id"].is_unique
assert reviewed_df["record_id"].notna().all()

# Verify original source and screening fields
IMMUTABLE_COLUMNS = [
    "record_id",
    "source_dataset",
    "source_record_id",
    "question",
    "answer",
    "crop",
    "country",
    "state_region",
    "local_area",
    "source_date",
    "matched_intents_v3",
    "relevance_status_v3",
    "candidate_intents",
    "proposed_primary_intent"
]

for column in IMMUTABLE_COLUMNS:
    assert original_df[column].equals(reviewed_df[column]), (
        f"Original data was modified in column: {column}"
    )

# Verify sidecar alignment
assert len(sidecar_df) == len(original_df)
assert sidecar_df["record_id"].equals(reviewed_df["record_id"])

for column in sidecar_df.columns:
    assert column in reviewed_df.columns
    assert sidecar_df[column].equals(reviewed_df[column]), (
        f"Sidecar mismatch in column: {column}"
    )

print("ALL STRUCTURAL CHECKS PASSED")
print("Original records preserved:", len(original_df))
print("Reviewed records:", len(reviewed_df))
print("Sidecar records:", len(sidecar_df))
print("Original source fields unchanged.")
print("Sidecar matches reviewed assessment.")

ALL STRUCTURAL CHECKS PASSED
Original records preserved: 522
Reviewed records: 522
Sidecar records: 522
Original source fields unchanged.
Sidecar matches reviewed assessment.


## 31.2 Creating Separate Dataset Collections

The reviewed dataset will now be separated according to its intended use.

### A. Intent classifier dataset
Contains records marked `provisional_pending_team_confirmation` in `classifier_label_status`.

The model will use:
- `question` as input
- `primary_intent` as target label

The original answers are retained for traceability but will not be used as classifier input.

### B. Rejected records
Contains all records with `review_decision = reject`.

### C. Pending review
Contains all records with `review_decision = needs_second_review`.

### D. Knowledge-base candidates
Contains records marked either:
- `candidate_pending_source_verification`
- `requires_expert_review`

These are candidate records only. They are not approved knowledge and must not be treated as verified agricultural advice.

### E. Duplicate-aware splitting
Near-duplicate questions must remain in the same dataset split to prevent similar questions from appearing in both training and testing data.

The original reviewed dataset will not be modified.

In [36]:
# Work on a copy. Never modify reviewed_df directly.
working_df = reviewed_df.copy()

# Near-duplicate clusters reported by the AI assessment
duplicate_clusters = [
    ["AQA_01138", "AQA_01140"],
    ["AQA_01428", "AQA_01825"],
    ["AQA_01470", "AQA_01471"],
    ["AQA_01496", "AQA_01507"],
    ["AQA_01298", "AQA_01299"],
    ["AQA_01278", "AQA_01865"],
    ["AQA_01301", "AQA_01302"],
    ["AQA_00302", "AQA_00312"],
    ["AQA_00907", "AQA_00909"],
    ["FCN_00775", "FCN_00947", "FCN_10828"],
    ["FCN_01108", "FCN_01110"],
    ["FCN_01679", "FCN_01987", "FCN_10896"],
    ["FCN_02248", "FCN_04054", "FCN_07148"],
    ["FCN_00608", "FCN_02568", "FCN_03832", "FCN_05622", "FCN_08812"],
    ["FCN_02993", "FCN_02603", "FCN_07118"],
    ["FCN_06273", "FCN_06562"],
    ["FCN_06611", "FCN_06612"],
    ["FCN_10494", "FCN_10496"],
    ["FCN_10622", "FCN_10626"],
    ["FCN_11083", "FCN_11087"],
    ["FCN_00058", "FCN_01512"],
    ["FCN_08623", "FCN_01037"],
    ["FCN_05251", "FCN_04980"],
    ["FCN_07659", "FCN_07675"]
]

# Map record IDs to duplicate groups
duplicate_lookup = {}

for number, cluster in enumerate(duplicate_clusters, start=1):
    group_id = f"NDG_{number:02d}"

    for record_id in cluster:
        if record_id in duplicate_lookup:
            raise ValueError(
                f"{record_id} appears in multiple duplicate groups."
            )

        duplicate_lookup[record_id] = group_id

# Ensure every reported duplicate ID exists
missing_duplicate_ids = (
    set(duplicate_lookup.keys()) - set(working_df["record_id"])
)

if missing_duplicate_ids:
    raise ValueError(
        f"Duplicate IDs missing from dataset: {missing_duplicate_ids}"
    )

# Add duplicate metadata
working_df["near_duplicate_group"] = (
    working_df["record_id"].map(duplicate_lookup).fillna("")
)

# Every record receives a split group.
# Near-duplicates share a group; unrelated records have individual groups.
working_df["split_group_id"] = working_df.apply(
    lambda row: (
        row["near_duplicate_group"]
        if row["near_duplicate_group"]
        else f"REC_{row['record_id']}"
    ),
    axis=1
)

# 1. Classifier candidate dataset
training_df = working_df[
    working_df["classifier_label_status"]
    == "provisional_pending_team_confirmation"
].copy()

# 2. Rejected records
rejected_df = working_df[
    working_df["review_decision"] == "reject"
].copy()

# 3. Records requiring another review
pending_df = working_df[
    working_df["review_decision"] == "needs_second_review"
].copy()

# 4. Knowledge-base candidates awaiting verification/review
knowledge_df = working_df[
    working_df["knowledge_base_status"].isin([
        "candidate_pending_source_verification",
        "requires_expert_review"
    ])
].copy()

# Validate that all records are accounted for by review decision
assert len(training_df) == 517
assert len(rejected_df) == 4
assert len(pending_df) == 212
assert len(knowledge_df) == 398

assert len(working_df) == len(original_df)
assert working_df["record_id"].is_unique

# Classifier target validation
VALID_INTENTS = {
    "crop_cultivation",
    "pest_management",
    "disease_management",
    "soil_management",
    "irrigation",
    "fertilizer_application",
    "harvesting",
    "out_of_scope"
}

assert set(training_df["primary_intent"]).issubset(VALID_INTENTS)
assert not (
    training_df["classifier_label_status"] == "unsuitable"
).any()

# Out-of-scope answers must not enter the knowledge candidate collection
assert not (
    knowledge_df["primary_intent"] == "out_of_scope"
).any()

assert not (
    knowledge_df["knowledge_base_status"] == "not_suitable"
).any()

print("DATASET COLLECTIONS CREATED")
print("--------------------------------")
print("Classifier candidates:", len(training_df))
print("Rejected records:", len(rejected_df))
print("Pending review:", len(pending_df))
print("Knowledge candidates:", len(knowledge_df))
print("Near-duplicate clusters:", len(duplicate_clusters))
print("Records in near-duplicate clusters:",
      len(duplicate_lookup))

DATASET COLLECTIONS CREATED
--------------------------------
Classifier candidates: 517
Rejected records: 4
Pending review: 212
Knowledge candidates: 398
Near-duplicate clusters: 24
Records in near-duplicate clusters: 55


### 31.3 Exporting Dataset Collections

We will export the four dataset collections into `data/processed/`.

The classifier dataset is a provisional candidate pool, not a final gold-standard training set. The model will use only the question and its intent label.

Near-duplicate group metadata is retained for grouped train/test splitting.

The knowledge-base collection is strictly a verification queue. None of its records is approved for trusted agricultural retrieval at this stage.

The review summary will contain counts and distributions to support dataset documentation.

In [37]:
# Output paths
training_output = PROCESSED_DIR / "edofarm_training_dataset.csv"
rejected_output = PROCESSED_DIR / "edofarm_rejected_records.csv"
pending_output = PROCESSED_DIR / "edofarm_pending_review.csv"
knowledge_output = PROCESSED_DIR / "edofarm_knowledge_candidates.csv"
summary_output = PROCESSED_DIR / "edofarm_review_summary.csv"

# Export dataset collections
training_df.to_csv(training_output, index=False, encoding="utf-8-sig")
rejected_df.to_csv(rejected_output, index=False, encoding="utf-8-sig")
pending_df.to_csv(pending_output, index=False, encoding="utf-8-sig")
knowledge_df.to_csv(knowledge_output, index=False, encoding="utf-8-sig")

# Build review summary
summary_rows = []

def add_distribution(df, population, dimension, column):
    total = len(df)

    for category, count in df[column].value_counts(
        dropna=False
    ).items():
        summary_rows.append({
            "population": population,
            "dimension": dimension,
            "category": str(category),
            "record_count": int(count),
            "share_within_population": round(
                count / total * 100, 2
            ) if total else 0
        })

def add_metric(population, metric, count, total):
    summary_rows.append({
        "population": population,
        "dimension": "summary_metric",
        "category": metric,
        "record_count": int(count),
        "share_within_population": round(
            count / total * 100, 2
        ) if total else 0
    })

# Overall review distributions
for column in [
    "review_decision",
    "primary_intent",
    "answer_quality",
    "nigerian_relevance",
    "source_check_status",
    "classifier_label_status",
    "knowledge_base_status",
    "source_dataset"
]:
    add_distribution(
        working_df,
        "all_reviewed_records",
        column,
        column
    )

# Classifier candidate distributions
add_distribution(
    training_df,
    "classifier_candidates",
    "primary_intent",
    "primary_intent"
)

add_distribution(
    training_df,
    "classifier_candidates",
    "review_decision",
    "review_decision"
)

# Knowledge candidate distributions
add_distribution(
    knowledge_df,
    "knowledge_candidates",
    "knowledge_base_status",
    "knowledge_base_status"
)

add_distribution(
    knowledge_df,
    "knowledge_candidates",
    "primary_intent",
    "primary_intent"
)

# Dataset summary metrics
add_metric("all_reviewed_records", "total_records", len(working_df), len(working_df))
add_metric("classifier_candidates", "classifier_candidate_records", len(training_df), len(training_df))
add_metric("rejected_records", "rejected_records", len(rejected_df), len(rejected_df))
add_metric("pending_review", "pending_review_records", len(pending_df), len(pending_df))
add_metric("knowledge_candidates", "knowledge_candidate_records", len(knowledge_df), len(knowledge_df))
add_metric("all_reviewed_records", "near_duplicate_cluster_count", len(duplicate_clusters), len(working_df))
add_metric("all_reviewed_records", "records_in_near_duplicate_clusters", len(duplicate_lookup), len(working_df))

review_summary_df = pd.DataFrame(summary_rows)

review_summary_df.to_csv(
    summary_output,
    index=False,
    encoding="utf-8-sig"
)

# Read-back verification
for path, expected_rows, expected_ids in [
    (training_output, len(training_df), training_df["record_id"]),
    (rejected_output, len(rejected_df), rejected_df["record_id"]),
    (pending_output, len(pending_df), pending_df["record_id"]),
    (knowledge_output, len(knowledge_df), knowledge_df["record_id"])
]:
    exported = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False
    )

    assert len(exported) == expected_rows
    assert exported["record_id"].equals(
        expected_ids.reset_index(drop=True)
    )
    assert "near_duplicate_group" in exported.columns
    assert "split_group_id" in exported.columns

print("EXPORTS COMPLETED SUCCESSFULLY")
print("--------------------------------")
for path in [
    training_output,
    rejected_output,
    pending_output,
    knowledge_output,
    summary_output
]:
    print(path.name, "|", path.stat().st_size, "bytes")

print("\nFinal row counts:")
print("Classifier candidates:", len(training_df))
print("Rejected:", len(rejected_df))
print("Pending:", len(pending_df))
print("Knowledge candidates:", len(knowledge_df))
print("Review summary rows:", len(review_summary_df))

EXPORTS COMPLETED SUCCESSFULLY
--------------------------------
edofarm_training_dataset.csv | 1178282 bytes
edofarm_rejected_records.csv | 6662 bytes
edofarm_pending_review.csv | 519829 bytes
edofarm_knowledge_candidates.csv | 943923 bytes
edofarm_review_summary.csv | 3815 bytes

Final row counts:
Classifier candidates: 517
Rejected: 4
Pending: 212
Knowledge candidates: 398
Review summary rows: 56


## 31.4 Classifier Dataset Distribution and Quality Audit

Before creating training, validation and test sets, we will inspect the distribution of the classifier candidates.

We will examine:

1. Number of records per intent.
2. Number of accepted versus pending-review records.
3. Answer-quality distribution.
4. Number and size of near-duplicate groups.
5. Whether near-duplicate groups contain conflicting intent labels.

For the initial baseline, we will use accepted records with provisional classifier labels.

Records requiring second review will remain available as a supplementary candidate pool, but will not enter the initial baseline until their labels are confirmed.

This produces a more conservative initial experiment.

In [38]:
# 1. Primary intent distribution
intent_distribution = (
    training_df["primary_intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="record_count")
)

print("CLASSIFIER CANDIDATE DISTRIBUTION")
display(intent_distribution)


# 2. Review decision distribution
print("\nREVIEW DECISIONS")
display(
    pd.crosstab(
        training_df["review_decision"],
        training_df["primary_intent"],
        margins=True
    )
)


# 3. Answer quality distribution
print("\nANSWER QUALITY")
display(training_df["answer_quality"].value_counts())


# 4. Create conservative baseline candidate pool
baseline_df = training_df[
    (training_df["review_decision"] == "accept") &
    (
        training_df["classifier_label_status"]
        == "provisional_pending_team_confirmation"
    )
].copy()

supplementary_df = training_df[
    training_df["review_decision"] == "needs_second_review"
].copy()

print("\nBASELINE CANDIDATE POOL")
print("Accepted classifier candidates:", len(baseline_df))
print("Supplementary candidates:", len(supplementary_df))


# 5. Inspect class distribution in baseline pool
print("\nBASELINE INTENT DISTRIBUTION")

baseline_distribution = (
    baseline_df["primary_intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="record_count")
)

display(baseline_distribution)


# 6. Check near-duplicate group consistency
group_audit = (
    training_df.groupby("split_group_id")
    .agg(
        records=("record_id", "count"),
        unique_intents=("primary_intent", "nunique"),
        unique_questions=("question", "nunique")
    )
    .reset_index()
)

conflicting_groups = group_audit[
    group_audit["unique_intents"] > 1
]

print("\nDUPLICATE GROUP AUDIT")
print("Total split groups:", len(group_audit))
print("Groups with multiple records:",
      (group_audit["records"] > 1).sum())
print("Groups with conflicting intents:", len(conflicting_groups))

if len(conflicting_groups) > 0:
    display(conflicting_groups)


# 7. Final checks
assert baseline_df["record_id"].is_unique
assert set(baseline_df["primary_intent"]).issubset(VALID_INTENTS)

print("\nDISTRIBUTION AUDIT COMPLETED")

CLASSIFIER CANDIDATE DISTRIBUTION


,intent,record_count
0,out_of_scope,76
1,fertilizer_application,75
2,disease_management,73
3,pest_management,73
4,soil_management,63
5,irrigation,60
6,harvesting,56
7,crop_cultivation,41



REVIEW DECISIONS


primary_intent,crop_cultivation,disease_management,fertilizer_application,harvesting,irrigation,out_of_scope,pest_management,soil_management,All
review_decision,,,,,,,,,
accept,17,40,40,33,32,75,25,44,306
needs_second_review,24,33,35,23,28,1,48,19,211
All,41,73,75,56,60,76,73,63,517



ANSWER QUALITY


answer_quality
clear_and_relevant                231
unclear_or_incomplete             185
not_applicable_classifier_only     75
potentially_misleading             26
Name: count, dtype: int64


BASELINE CANDIDATE POOL
Accepted classifier candidates: 306
Supplementary candidates: 211

BASELINE INTENT DISTRIBUTION


,intent,record_count
0,out_of_scope,75
1,soil_management,44
2,disease_management,40
3,fertilizer_application,40
4,harvesting,33
5,irrigation,32
6,pest_management,25
7,crop_cultivation,17



DUPLICATE GROUP AUDIT
Total split groups: 486
Groups with multiple records: 24
Groups with conflicting intents: 0

DISTRIBUTION AUDIT COMPLETED


## 31.5 Creating Training, Validation and Test Splits

The accepted classifier candidate dataset contains 306 records across eight intents.

The dataset is moderately imbalanced, so stratification will be used to preserve the intent distribution as closely as possible.

We will use StratifiedGroupKFold to ensure that records belonging to the same near-duplicate group remain in the same partition.

The planned split is approximately:

- 64% training
- 16% validation
- 20% testing

The test set will remain untouched during model training and model selection.

Only the question and primary intent will be used as model input and target. Answers will not be provided to the intent classifier.

### Installing Scikit-learn

Scikit-learn provides the machine-learning algorithms and utilities required for our intent classification experiment.

We will install it in the current Jupyter environment.

In [40]:
%pip install scikit-learn

  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 513.0 kB/s eta 0:00:16
   -- ------------------------------------- 0.5/8.4 MB 513.0 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/8.4 MB 562.8 kB/s eta 0:00:14
   --- ------------------------------------ 0.8/8.4 MB 562.8 kB/s eta 0:00:14
   ----- ---------------------------------- 1.0/8.4 MB 609.8 kB/s eta 0:00:13
   ------ --------------------------------- 1.3/8.4 MB 643.9 kB/s eta 0:00:11
   ------- -------------------------------- 1.6/8.4 MB 676.4 kB/s eta 0:00:11
   ------- ---------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd

# Work only with accepted classifier candidates
split_df = baseline_df.copy().reset_index(drop=True)

# Confirm the required columns
required_columns = [
    "record_id",
    "question",
    "primary_intent",
    "split_group_id",
    "near_duplicate_group",
    "source_dataset",
    "review_decision",
    "answer_quality"
]

assert all(col in split_df.columns for col in required_columns)

# Check that each intent has enough independent groups
group_counts = (
    split_df.groupby("primary_intent")["split_group_id"]
    .nunique()
)

print("Independent groups per intent:")
display(group_counts)

if group_counts.min() < 5:
    raise ValueError(
        "At least one intent has fewer than five independent groups. "
        "A five-fold grouped split may not be reliable."
    )

y = split_df["primary_intent"].to_numpy()
groups = split_df["split_group_id"].to_numpy()

expected_labels = set(split_df["primary_intent"])

# Find a deterministic split with all labels represented
split_found = False

for seed in range(42, 142):

    outer_splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    train_val_idx, test_idx = next(
        outer_splitter.split(split_df["question"], y, groups)
    )

    train_val_df = split_df.iloc[train_val_idx].copy()

    inner_splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed + 1000
    )

    train_idx, val_idx = next(
        inner_splitter.split(
            train_val_df["question"],
            train_val_df["primary_intent"],
            train_val_df["split_group_id"]
        )
    )

    train_df = train_val_df.iloc[train_idx].copy()
    val_df = train_val_df.iloc[val_idx].copy()
    test_df = split_df.iloc[test_idx].copy()

    # Check that every intent appears in all partitions
    coverage_ok = all(
        set(frame["primary_intent"]) == expected_labels
        for frame in [train_df, val_df, test_df]
    )

    # Check group leakage
    train_groups = set(train_df["split_group_id"])
    val_groups = set(val_df["split_group_id"])
    test_groups = set(test_df["split_group_id"])

    no_group_leakage = (
        train_groups.isdisjoint(val_groups)
        and train_groups.isdisjoint(test_groups)
        and val_groups.isdisjoint(test_groups)
    )

    if coverage_ok and no_group_leakage:
        split_found = True
        break

if not split_found:
    raise RuntimeError(
        "Could not find a grouped split with all eight intents "
        "represented in every partition. Inspect the group distribution."
    )

print("Grouped stratified split generated.")
print("Random seed:", seed)
print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))

Independent groups per intent:


primary_intent
crop_cultivation          17
disease_management        36
fertilizer_application    39
harvesting                27
irrigation                30
out_of_scope              71
pest_management           23
soil_management           44
Name: split_group_id, dtype: int64

Grouped stratified split generated.
Random seed: 42
Training records: 196
Validation records: 49
Test records: 61


In [42]:
# Verify every intent is represented
for name, frame in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name} distribution:")
    display(frame["primary_intent"].value_counts())

# Verify no leakage
assert set(train_df["split_group_id"]).isdisjoint(
    set(val_df["split_group_id"])
)

assert set(train_df["split_group_id"]).isdisjoint(
    set(test_df["split_group_id"])
)

assert set(val_df["split_group_id"]).isdisjoint(
    set(test_df["split_group_id"])
)

# Create one combined split manifest
split_exports = []

for split_name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    export = frame[required_columns].copy()
    export["split"] = split_name
    split_exports.append(export)

classifier_splits_df = pd.concat(
    split_exports,
    ignore_index=True
)

# Export the three partitions
train_df[required_columns].to_csv(
    PROCESSED_DIR / "classifier_train.csv",
    index=False,
    encoding="utf-8-sig"
)

val_df[required_columns].to_csv(
    PROCESSED_DIR / "classifier_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df[required_columns].to_csv(
    PROCESSED_DIR / "classifier_test.csv",
    index=False,
    encoding="utf-8-sig"
)

classifier_splits_df.to_csv(
    PROCESSED_DIR / "edofarm_classifier_splits.csv",
    index=False,
    encoding="utf-8-sig"
)

# Final checks
assert len(classifier_splits_df) == len(baseline_df)
assert classifier_splits_df["record_id"].is_unique
assert set(classifier_splits_df["record_id"]) == set(
    baseline_df["record_id"]
)

print("\nSPLIT EXPORTS COMPLETED")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(classifier_splits_df))
print("Group leakage: NONE")
print("Missing intent classes: NONE")


Training distribution:


primary_intent
out_of_scope              48
soil_management           28
fertilizer_application    26
disease_management        25
harvesting                22
irrigation                20
pest_management           16
crop_cultivation          11
Name: count, dtype: int64


Validation distribution:


primary_intent
out_of_scope              12
disease_management         7
soil_management            7
fertilizer_application     6
irrigation                 5
harvesting                 5
pest_management            4
crop_cultivation           3
Name: count, dtype: int64


Test distribution:


primary_intent
out_of_scope              15
soil_management            9
fertilizer_application     8
disease_management         8
irrigation                 7
harvesting                 6
pest_management            5
crop_cultivation           3
Name: count, dtype: int64


SPLIT EXPORTS COMPLETED
Training: 196
Validation: 49
Test: 61
Total: 306
Group leakage: NONE
Missing intent classes: NONE
